# 06 · Convolutional Neural Networks

**Part III — Computer vision** · Estimated time: ~150 min

## Learning objectives

- Explain why fully connected layers are a poor fit for images and which inductive biases (locality, weight sharing, translation equivariance) convolutions add.
- Implement 1-D and 2-D cross-correlation from scratch, including padding, stride and multiple channels, and verify it numerically against `F.conv2d`.
- Compute output sizes, parameter counts, multiply-accumulate costs and receptive fields of convolutional layers, and explain convolution as a matrix multiplication (im2col).
- Use pooling, strided convolutions, global average pooling, 1x1 convolutions, depthwise-separable convolutions and transposed convolutions, and know what each one costs.
- Build, train and inspect LeNet-5 on MNIST.
- Train a VGG-style network with BatchNorm and `transforms.v2` augmentation on CIFAR-10.
- Reproduce the degradation problem of plain deep networks, derive the residual block, and train a ResNet on CIFAR-10.
- Visualize filters, feature maps and occlusion-sensitivity maps to understand what a CNN has learned.

## Prerequisites

- Notebook 03 · Tensors and Autograd (tensor shapes, broadcasting, `autograd`)
- Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization (SGD with momentum, BatchNorm, weight decay, learning-rate schedules, data augmentation as regularization)
- Notebook 05 · The PyTorch Workflow and Best Practices (`Dataset`/`DataLoader`, `nn.Module`, training and evaluation loops)

## Contents

1. Why convolutions
2. The convolution operation
3. Pooling, strides and downsampling
4. LeNet-5 on MNIST
5. CIFAR-10 and a VGG-style network
6. Residual networks
7. A tour of architectures
8. Understanding CNNs

Exercises · Summary · Further reading

In [ ]:
import math, os, random, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
from torchvision import datasets
from torchvision.transforms import v2
from sklearn.datasets import load_sample_image
from sklearn.metrics import confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

# Mixed precision is used automatically on a GPU (Notebook 14 · Efficient Training and Scaling
# explains it in depth); on a CPU every autocast / GradScaler call below is a no-op.
USE_AMP = device.type == "cuda"


def to_np(t):
    """Detach a tensor, move it to the CPU and hand it to NumPy (for plotting)."""
    return np.asarray(t.detach().cpu())


def count_params(model):
    """Number of trainable parameters of a module."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("FAST_RUN =", FAST_RUN)

## 1. Why convolutions

### 1.1 Images are tensors

A grayscale image is a grid of brightness values, a matrix of shape $(H, W)$. A colour image has three such
grids, one each for red, green and blue, and PyTorch stores it as a tensor of shape $(C, H, W)$ with
$C = 3$ **channels**. A mini-batch of images adds a leading batch dimension: $(N, C, H, W)$. This
**channels-first** convention (as opposed to NumPy's and matplotlib's channels-last $(H, W, C)$) is the
single most frequent source of shape errors in vision code, so we make it explicit from the first cell.

Every layer in this notebook consumes and produces tensors of shape $(N, C, H, W)$. Convolutions change
$C$ (the number of feature maps) and optionally $H$ and $W$; pooling layers change $H$ and $W$ only; the
final classifier flattens whatever is left into a vector. Keeping track of these four numbers is most of
the work of designing a CNN, and we will build helper functions that do it for us.

In [ ]:
china = load_sample_image("china.jpg")                       # NumPy array, (H, W, C), uint8
print("NumPy image  (H, W, C):", china.shape, china.dtype)

img = torch.from_numpy(china).permute(2, 0, 1).float() / 255.0   # (C, H, W), values in [0, 1]
print("PyTorch image (C, H, W):", tuple(img.shape), img.dtype)

gray = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]      # luminance, shape (H, W)
print("grayscale     (H, W)   :", tuple(gray.shape))

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
axes[0].imshow(china); axes[0].set_title("RGB image")
for c, name in enumerate(["red", "green", "blue"]):
    axes[c + 1].imshow(to_np(img[c]), cmap="gray", vmin=0, vmax=1)
    axes[c + 1].set_title(f"channel {c}: {name}")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

### 1.2 What goes wrong with fully connected layers

The multi-layer perceptrons of Notebooks 02–05 take a flat vector as input. To feed them an image we
flatten it: the $427 \times 640 \times 3$ photograph above becomes a vector with $819{,}840$ entries. A
single hidden layer with a modest $1{,}024$ units then needs $819{,}840 \times 1{,}024 \approx 8.4 \times 10^8$
weights: more than three gigabytes in `float32`, for one layer, before we have learned anything.

The parameter count is only the first problem. Flattening also throws away the *geometry* of the image:
a dense layer has no notion that pixel $(i, j)$ is next to pixel $(i, j+1)$, or that a cat in the top-left
corner is the same object as a cat in the bottom-right corner. It would have to learn "what a cat looks
like" separately for every position, from data, with no help from the architecture. Dense networks *can*
be trained on small images (we did it on MNIST), but they overfit badly and do not scale.

In [ ]:
H, W, C = china.shape
n_inputs = C * H * W
hidden = 1024
dense_params = n_inputs * hidden + hidden

conv = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1)
conv_params = count_params(conv)

print(f"flattened input size                       : {n_inputs:>12,}")
print(f"one dense layer with {hidden} hidden units   : {dense_params:>12,} parameters "
      f"({dense_params * 4 / 1e9:.2f} GB in float32)")
print(f"one 3x3 conv layer with 64 output channels : {conv_params:>12,} parameters")
print(f"ratio                                      : {dense_params / conv_params:>12,.0f} x")

### 1.3 Three inductive biases

A **convolutional layer** replaces the dense weight matrix with a small set of **kernels** (also called
filters) that are slid across the image. This design encodes three assumptions about images, which the
literature calls **inductive biases** because they bias the model towards solutions that are plausible for
this kind of data:

1. **Locality.** Each output value depends only on a small neighbourhood of the input (a $3 \times 3$ or
   $5 \times 5$ window). Edges, corners and textures are local phenomena, so this is the right place to
   start; larger structures are captured by stacking layers (Section 2.6 on receptive fields).
2. **Weight sharing.** The *same* kernel is applied at every position. A vertical-edge detector is useful
   everywhere in the image, so there is no reason to learn it once per location. This is what makes the
   parameter count independent of the image size: the convolution above has $64 \times 3 \times 3 \times 3 + 64 = 1{,}792$
   parameters whether the image is $28 \times 28$ or $4{,}000 \times 4{,}000$.
3. **Translation equivariance.** Shifting the input shifts the output by the same amount:
   $\mathrm{conv}(\mathrm{shift}(x)) = \mathrm{shift}(\mathrm{conv}(x))$ (exactly, away from the image
   border). Combined with pooling and the final global aggregation this becomes approximate translation
   *invariance*: a cat is a cat wherever it sits.

These biases are not free: they are assumptions. When they hold (natural images, audio spectrograms,
board-game positions) CNNs learn from far less data than unconstrained models. When they do not (tabular
data, graphs), other architectures are better. Notebook 07 · Transfer Learning and Modern Vision returns
to this trade-off when we compare CNNs with Vision Transformers, which have weaker inductive biases and
correspondingly need more data.

Let us verify the equivariance property numerically with a randomly initialised convolution. We shift the
image by five pixels (with `torch.roll`, which wraps around) before and after the convolution and compare
the two results away from the border, where padding and wrap-around make them differ.

In [ ]:
torch.manual_seed(SEED)
conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
x = gray[None, None, :128, :128]                       # (N=1, C=1, H=128, W=128)
shift = 5

with torch.no_grad():
    shift_then_conv = conv(torch.roll(x, shifts=(shift, shift), dims=(2, 3)))
    conv_then_shift = torch.roll(conv(x), shifts=(shift, shift), dims=(2, 3))

interior = (slice(None), slice(None), slice(shift + 1, -1), slice(shift + 1, -1))
diff = (shift_then_conv - conv_then_shift)[interior].abs().max().item()
print(f"max |conv(shift(x)) - shift(conv(x))| away from the border: {diff:.2e}")

The difference is zero up to floating-point rounding: the convolution "does not know" where in the image
it is being applied. A dense layer has no such property.

> **Key idea.** A convolutional layer is a linear layer with two constraints, locality and weight sharing.
> The constraints cut the parameter count by orders of magnitude and give translation equivariance for free.

A convolutional neural network (CNN) is then a stack of such layers, each followed by a non-linearity,
interleaved with downsampling operations that gradually trade spatial resolution for channel depth, and
finished by a small classifier. The rest of this notebook builds up each ingredient and then assembles
them into LeNet (1998), a VGG-style network (2014) and a ResNet (2015).

## 2. The convolution operation

### 2.1 One dimension first: cross-correlation

Take a signal $x \in \mathbb{R}^n$ and a kernel $w \in \mathbb{R}^k$ with $k \le n$. The
**cross-correlation** of $x$ with $w$ slides the kernel along the signal and, at each position $i$, takes the
dot product of the kernel with the window of the signal under it:

$$
y[i] = \sum_{m=0}^{k-1} x[i + m]\, w[m], \qquad i = 0, 1, \dots, n - k .
$$

The output has $n - k + 1$ entries when no padding is used (the "valid" positions, where the kernel fits
entirely inside the signal).

The **convolution** of mathematics and signal processing is almost the same operation with the kernel
*flipped*: $(x * w)[i] = \sum_m x[i - m]\, w[m]$. Flipping makes convolution commutative and associative,
which matters in signal processing. In deep learning it does not matter at all: the kernel is *learned*,
and a network that learns $w$ under one convention learns the flipped $w$ under the other. Every deep
learning framework therefore implements cross-correlation and calls it convolution, and we follow that
convention. The cell below shows the difference once, so that it never confuses you again.

In [ ]:
def corr1d(x, w):
    """Valid 1-D cross-correlation: y[i] = sum_m x[i + m] * w[m]."""
    n, k = len(x), len(w)
    y = np.zeros(n - k + 1)
    for i in range(n - k + 1):
        y[i] = np.dot(x[i:i + k], w)
    return y


x = np.array([1., 2., 3., 4., 5., 6., 7., 8.])
w = np.array([1., 0., -1.])
print("x                    :", x)
print("w                    :", w)
print("corr1d (ours)        :", corr1d(x, w))
print("np.correlate (valid) :", np.correlate(x, w, mode="valid"))
print("np.convolve  (valid) :", np.convolve(x, w, mode="valid"), "<- true convolution flips the kernel")
print("np.convolve, w[::-1] :", np.convolve(x, w[::-1], mode="valid"))

xt = torch.tensor(x, dtype=torch.float32).view(1, 1, -1)    # (N, C_in, L)
wt = torch.tensor(w, dtype=torch.float32).view(1, 1, -1)    # (C_out, C_in, k)
print("F.conv1d             :", F.conv1d(xt, wt).flatten().tolist())

Our loop, `np.correlate` and `F.conv1d` agree: with the kernel $(1, 0, -1)$ every output is
$x[i] - x[i+2] = -2$, a (negative) finite-difference estimate of the slope. `np.convolve` gives $+2$
because it flips the kernel. Note the shapes PyTorch expects: the signal is `(batch, in_channels, length)`
and the kernel is `(out_channels, in_channels, kernel_size)`; the 2-D case simply adds one more spatial
dimension to each.

### 2.2 Two dimensions

For an image $X \in \mathbb{R}^{H \times W}$ and a kernel $K \in \mathbb{R}^{k_h \times k_w}$ the
2-D cross-correlation is

$$
Y[i, j] = \sum_{a=0}^{k_h - 1} \sum_{b=0}^{k_w - 1} X[i + a,\, j + b]\; K[a, b],
\qquad Y \in \mathbb{R}^{(H - k_h + 1) \times (W - k_w + 1)} .
$$

Each output pixel is the dot product of the kernel with the patch of the image under it. We implement
this with two explicit Python loops over the output positions. The implementation is hopelessly slow for
real images (we will measure how slow in Section 2.7) but it is the clearest possible statement of what a
convolution computes, and we will use it as the reference against which every faster version is checked.
We include `stride` and `padding` right away; they are explained in the next subsection.

In [ ]:
def corr2d(x, w, stride=1, padding=0):
    """2-D cross-correlation of a single-channel image x (H, W) with a kernel w (kh, kw).

    Zero-pads `padding` pixels on every side, then slides the kernel with step `stride`.
    Returns an array of shape (H_out, W_out).
    """
    if padding > 0:
        x = np.pad(x, padding)
    H, W = x.shape
    kh, kw = w.shape
    H_out = (H - kh) // stride + 1
    W_out = (W - kw) // stride + 1
    y = np.zeros((H_out, W_out), dtype=np.float64)
    for i in range(H_out):
        for j in range(W_out):
            r, c = i * stride, j * stride
            y[i, j] = np.sum(x[r:r + kh, c:c + kw] * w)
    return y


# A toy image: white (1) on the left and right, a black (0) band in the middle.
X = np.ones((6, 8))
X[:, 2:6] = 0.0
K = np.array([[1.0, -1.0]])           # horizontal finite difference
print("X =\n", X)
print("\ncorr2d(X, K) =\n", corr2d(X, K))
print("\ncorr2d(X.T, K) =\n", corr2d(X.T, K))

The kernel $K = (1, -1)$ responds with $+1$ where the image goes from white to black (column 1), with
$-1$ where it goes from black to white (column 5), and with $0$ everywhere else: it is a **vertical-edge
detector**. Applied to the transposed image, whose edges are horizontal, it returns all zeros. This is the
whole idea of a learned convolutional layer: a kernel is a small pattern, and the output map says *where*
that pattern occurs.

Before going further we check our implementation against PyTorch's `F.conv2d` for several kernel sizes,
strides and paddings. `F.conv2d` wants a 4-D input `(N, C_in, H, W)` and a 4-D weight
`(C_out, C_in, kh, kw)`, so we add two singleton dimensions with `[None, None]`.

In [ ]:
def check_corr2d(x_np, w_np, stride, padding):
    ours = corr2d(x_np, w_np, stride=stride, padding=padding)
    ref = F.conv2d(torch.tensor(x_np)[None, None], torch.tensor(w_np)[None, None],
                   stride=stride, padding=padding)[0, 0]
    return np.abs(ours - to_np(ref)).max(), ours.shape


rng = np.random.default_rng(SEED)
x_np = rng.standard_normal((9, 11)).astype(np.float32)
for k, s, p in [(3, 1, 0), (3, 1, 1), (5, 2, 2), (4, 3, 1), (2, 2, 0)]:
    w_np = rng.standard_normal((k, k)).astype(np.float32)
    err, shape = check_corr2d(x_np, w_np, s, p)
    print(f"k={k} stride={s} padding={p}: output {str(shape):8s} max |ours - F.conv2d| = {err:.1e}")

The errors are at the level of `float32` rounding ($10^{-6}$ or below), so the loop implementation and
PyTorch compute the same thing, for every combination of stride and padding.

### 2.3 Padding, stride, dilation and the output size

Three hyperparameters control the geometry of a convolution.

**Padding** adds $p$ rows/columns of zeros around the input before sliding the kernel. Without padding
every layer shrinks the image by $k - 1$ pixels and the border pixels are seen by fewer kernel positions
than interior pixels. With $p = (k-1)/2$ (for odd $k$) the output has the same size as the input, which is
what almost every modern architecture uses ("same" padding: $p = 1$ for $3 \times 3$ kernels, $p = 2$ for
$5 \times 5$, $p = 3$ for $7 \times 7$).

**Stride** $s$ moves the kernel $s$ pixels at a time instead of one. A stride of 2 halves the output
resolution and quarters the cost; it is the modern way of downsampling (Section 3.2).

**Dilation** $d$ inserts $d - 1$ gaps between the kernel taps, so a $3 \times 3$ kernel with $d = 2$ covers
a $5 \times 5$ window with 9 parameters. It enlarges the receptive field without adding parameters and is
common in segmentation networks (Exercise 1 implements it).

Let us derive the output size along one axis. After padding the input has length $n + 2p$. A kernel with
dilation $d$ spans $d(k - 1) + 1$ positions (its *effective* size). The first kernel placement starts at
index 0; the last placement must start at an index $\le n + 2p - (d(k-1) + 1)$; placements are $s$ apart.
The number of placements is therefore

$$
n_{\text{out}} = \left\lfloor \frac{n + 2p - d(k - 1) - 1}{s} \right\rfloor + 1 .
$$

With $d = 1$ this is the familiar $\lfloor (n + 2p - k)/s \rfloor + 1$. The floor matters: if the kernel
does not fit an integer number of times, the last few input pixels are simply ignored. We wrap the formula
in a helper and check it against `nn.Conv2d` for the configurations you will meet most often.

In [ ]:
def conv_out_size(n, k, stride=1, padding=0, dilation=1):
    """Spatial output size of a convolution (or pooling) along one axis."""
    return (n + 2 * padding - dilation * (k - 1) - 1) // stride + 1


configs = [("3x3, s=1, p=0  ('valid')",     3, 1, 0, 1),
           ("3x3, s=1, p=1  ('same')",      3, 1, 1, 1),
           ("5x5, s=1, p=2  ('same')",      5, 1, 2, 1),
           ("3x3, s=2, p=1  (halve)",       3, 2, 1, 1),
           ("7x7, s=2, p=3  (ResNet stem)", 7, 2, 3, 1),
           ("3x3, s=1, p=0, d=2 (dilated)", 3, 1, 0, 2),
           ("3x3, s=1, p=2, d=2 ('same')",  3, 1, 2, 2),
           ("1x1, s=1, p=0",                1, 1, 0, 1),
           ("2x2, s=2, p=0  (like pooling)", 2, 2, 0, 1)]

n_in = 32
x = torch.randn(1, 1, n_in, n_in)
print(f"input size {n_in}\n{'configuration':34s} {'formula':>8s} {'nn.Conv2d':>10s}")
for name, k, s, p, d in configs:
    predicted = conv_out_size(n_in, k, stride=s, padding=p, dilation=d)
    actual = nn.Conv2d(1, 1, k, stride=s, padding=p, dilation=d)(x).shape[-1]
    print(f"{name:34s} {predicted:8d} {actual:10d}")

Two rows deserve a comment. The "halve" row shows the standard recipe for reducing resolution with a
convolution: $k = 3$, $s = 2$, $p = 1$ maps 32 to 16 (and, in general, $n$ to $\lceil n/2 \rceil$). The
"ResNet stem" row is the very first layer of the ImageNet ResNets: a $7 \times 7$ kernel with stride 2,
which turns a $224 \times 224$ image into $112 \times 112$.

> **Common pitfall.** Forgetting the floor. A $3 \times 3$ stride-2 convolution without padding maps
> $32 \to 15$, not $16$, and the last row and column of the input are never seen by the kernel.

### 2.4 Multiple input and output channels

Real inputs have several channels (3 for RGB, or 64 feature maps produced by a previous layer), and a
layer produces several output channels, one per kernel. For an input $X \in \mathbb{R}^{C_{\text{in}} \times H \times W}$
a single output channel $o$ is computed by a kernel $K_o \in \mathbb{R}^{C_{\text{in}} \times k_h \times k_w}$: one
2-D cross-correlation *per input channel*, summed over channels, plus a bias:

$$
Y[o] = b_o + \sum_{c=0}^{C_{\text{in}}-1} X[c] \star K[o, c] .
$$

Stacking the $C_{\text{out}}$ kernels gives the 4-D weight tensor of shape
$(C_{\text{out}}, C_{\text{in}}, k_h, k_w)$ that you saw in `nn.Conv2d`. The number of parameters is

$$
\#\text{params} = C_{\text{out}} \cdot C_{\text{in}} \cdot k_h \cdot k_w + C_{\text{out}},
$$

independent of $H$ and $W$. Note that a $3 \times 3$ kernel with 64 input channels is not a $3 \times 3$
object at all: it is a $64 \times 3 \times 3 = 576$-dimensional pattern that mixes information across
*all* input channels at each location. Only the very first layer's kernels can be displayed as little
pictures (Section 4.4 and 8.1).

We extend the from-scratch implementation to channels and batches by looping (slowly) and check it against
`F.conv2d` once more, this time with stride and padding, a bias, a batch of two images, three input and
four output channels.

In [ ]:
def corr2d_multi(x, w, b=None, stride=1, padding=0):
    """x: (C_in, H, W); w: (C_out, C_in, kh, kw); b: (C_out,) or None -> (C_out, H_out, W_out)."""
    C_out, C_in = w.shape[:2]
    assert x.shape[0] == C_in, "input channels must match the kernel's second dimension"
    outs = []
    for o in range(C_out):
        acc = sum(corr2d(x[c], w[o, c], stride=stride, padding=padding) for c in range(C_in))
        if b is not None:
            acc = acc + b[o]
        outs.append(acc)
    return np.stack(outs)


def conv2d_batch(x, w, b=None, stride=1, padding=0):
    """x: (N, C_in, H, W) -> (N, C_out, H_out, W_out), looping over the batch."""
    return np.stack([corr2d_multi(xi, w, b, stride, padding) for xi in x])


x_np = rng.standard_normal((2, 3, 8, 8)).astype(np.float32)
w_np = rng.standard_normal((4, 3, 3, 3)).astype(np.float32)
b_np = rng.standard_normal(4).astype(np.float32)

ours = conv2d_batch(x_np, w_np, b_np, stride=2, padding=1)
ref = F.conv2d(torch.tensor(x_np), torch.tensor(w_np), torch.tensor(b_np), stride=2, padding=1)
print("output shape:", ours.shape, "| max |ours - F.conv2d| =", f"{np.abs(ours - to_np(ref)).max():.1e}")
print(f"parameters: C_out*C_in*k*k + C_out = 4*3*3*3 + 4 = {w_np.size + b_np.size}")

### 2.5 Hand-designed kernels on a real image

Before neural networks, computer vision engineers designed kernels by hand. Applying a few classics to the
photograph makes the "pattern detector" interpretation concrete and shows what the first layer of a CNN
typically rediscovers on its own:

- a **box blur** and a **Gaussian blur** average neighbouring pixels (low-pass filters);
- the **Sobel** kernels estimate the horizontal and vertical image gradient; their magnitude
  $\sqrt{G_x^2 + G_y^2}$ is an edge map;
- the **Laplacian** approximates the second derivative and lights up at edges and blobs;
- **sharpen** subtracts the Laplacian from the image (the kernel below is identity minus Laplacian), which
  boosts the fine detail that a blur removes.

We use `F.conv2d` rather than our loop here (the picture has $427 \times 640$ pixels); we have just shown
that the two are equivalent.

In [ ]:
kernels = {
    "box blur 5x5":   np.full((5, 5), 1 / 25),
    "Gaussian 5x5":   np.outer([1, 4, 6, 4, 1], [1, 4, 6, 4, 1]) / 256,
    "Sobel x":        np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "Sobel y":        np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
    "Laplacian":      np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]]),
    "sharpen":        np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
}


def apply_kernel(image_hw, k):
    kt = torch.tensor(np.asarray(k, dtype=np.float32))[None, None]
    return F.conv2d(image_hw[None, None], kt, padding=kt.shape[-1] // 2)[0, 0]


responses = {name: apply_kernel(gray, k) for name, k in kernels.items()}
responses["gradient magnitude"] = torch.sqrt(responses["Sobel x"] ** 2 + responses["Sobel y"] ** 2)

fig, axes = plt.subplots(2, 4, figsize=(14, 6.5))
axes = list(axes.flat)
axes[0].imshow(to_np(gray), cmap="gray"); axes[0].set_title("grayscale image")
for ax, (name, out) in zip(axes[1:], responses.items()):
    ax.imshow(to_np(out), cmap="gray"); ax.set_title(name)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

The blurs remove fine detail, the Sobel kernels respond to vertical and horizontal structure respectively
(look at the roof tiles versus the pillars), and the gradient magnitude is a clean edge map. A trained CNN
does not receive these kernels; it learns them, and when we look at the first layer of a trained network
in Sections 4.4 and 8.1 we will find oriented edge detectors and colour-contrast detectors that look
remarkably like these.

> **Try it.** Replace the Sobel kernels with the simpler central difference `[[-1, 0, 1]]` and its
> transpose, or scale the Laplacian by $-1$. Which changes affect the picture and which only flip its sign?

### 2.6 Receptive fields

A single $3 \times 3$ convolution sees a $3 \times 3$ patch of the input. Two of them stacked see
$5 \times 5$: each output pixel of the second layer depends on a $3 \times 3$ patch of the first layer's
output, and each of those depends on a $3 \times 3$ patch of the input, so the union is $5 \times 5$. The
region of the *input* that can influence one output pixel is called the **receptive field** of that
layer. Understanding how it grows is essential: a layer can only recognise an object if its receptive
field is large enough to contain it.

Let $r_l$ be the receptive-field size after layer $l$ and $j_l$ the **jump**, the distance in input pixels
between two neighbouring pixels of layer $l$'s output (the product of all strides so far). Starting from
$r_0 = 1$ and $j_0 = 1$, a layer with kernel size $k_l$ and stride $s_l$ gives

$$
r_l = r_{l-1} + (k_l - 1)\, j_{l-1}, \qquad j_l = j_{l-1} \cdot s_l .
$$

The first equation says that each additional kernel tap reaches $j_{l-1}$ further into the input on each
side. A dilated kernel is handled by using its effective size $d(k-1)+1$ in place of $k$; padding does not
change the receptive field (it only shifts where it is centred). Pooling layers are convolutions as far as
this formula is concerned. The helper below tabulates $r_l$ and $j_l$ for a VGG-like stack, and then we
*measure* the receptive field with autograd by asking which input pixels receive a non-zero gradient from a
single output pixel.

In [ ]:
def receptive_field(layers):
    """Receptive-field size and jump after each layer.

    layers: list of (kernel_size, stride) pairs in forward order. For a dilated kernel
    pass the effective size d*(k-1)+1. Returns a list of (rf, jump) tuples.
    """
    rf, jump, rows = 1, 1, []
    for k, s in layers:
        rf = rf + (k - 1) * jump
        jump = jump * s
        rows.append((rf, jump))
    return rows


vgg_like = [("conv 3x3", 3, 1), ("conv 3x3", 3, 1), ("pool 2x2", 2, 2),
            ("conv 3x3", 3, 1), ("conv 3x3", 3, 1), ("pool 2x2", 2, 2),
            ("conv 3x3", 3, 1), ("conv 3x3", 3, 1), ("pool 2x2", 2, 2)]

print(f"{'layer':10s} {'RF':>4s} {'jump':>5s}")
for (name, k, s), (rf, j) in zip(vgg_like, receptive_field([(k, s) for _, k, s in vgg_like])):
    print(f"{name:10s} {rf:4d} {j:5d}")


def measure_receptive_field(layers, n_in=64):
    """Measure the receptive field of the last layer with autograd (all-ones kernels, avg-pooling)."""
    mods = []
    for name, k, s in layers:
        if name.startswith("pool"):
            mods.append(nn.AvgPool2d(k, stride=s))
        else:
            conv = nn.Conv2d(1, 1, k, stride=s, padding=k // 2, bias=False)
            nn.init.constant_(conv.weight, 1.0)
            mods.append(conv)
    net = nn.Sequential(*mods)
    x = torch.zeros(1, 1, n_in, n_in, requires_grad=True)
    out = net(x)
    c = out.shape[-1] // 2                                   # a central output pixel
    (grad,) = torch.autograd.grad(out[0, 0, c, c], x)
    rows = torch.nonzero(grad[0, 0].abs().sum(dim=1)).flatten()
    return int(rows.max() - rows.min() + 1)


print("\nmeasured receptive field of the last layer:", measure_receptive_field(vgg_like))

The formula and the measurement agree: after three stages of two $3 \times 3$ convolutions and a pooling
layer, one output pixel sees a $36 \times 36$ patch, larger than a whole CIFAR-10 image ($32 \times 32$).
Notice how the jump does the work: after two poolings each $3 \times 3$ kernel adds $2 \times 4 = 8$ pixels
to the receptive field instead of 2. This is why deep networks alternate convolutions and downsampling
rather than using ever larger kernels: it is the cheapest way to make the receptive field grow.

> **Going deeper.** The *theoretical* receptive field computed here is an upper bound. Luo et al. (2016)
> showed that the *effective* receptive field, the region that actually has a noticeable influence on the
> output, is much smaller and roughly Gaussian-shaped, because pixels near the centre reach the output
> through many more paths than pixels at the edge. Residual connections and training both change its
> shape. Exercise 2 computes theoretical receptive fields for the networks we train later.

### 2.7 Convolution as matrix multiplication (im2col)

How do frameworks compute convolutions fast? One classic answer is to turn the convolution into a single
large matrix multiplication, an operation for which every hardware vendor ships a heavily optimised
routine (GEMM). The trick is called **im2col** ("image to columns"):

1. extract every $C_{\text{in}} \times k \times k$ patch that the kernel will visit and flatten it into a
   column vector of length $C_{\text{in}} k^2$; there are $L = H_{\text{out}} W_{\text{out}}$ such patches,
   so they form a matrix $\mathbf{P} \in \mathbb{R}^{C_{\text{in}} k^2 \times L}$;
2. flatten the weights into a matrix $\mathbf{W} \in \mathbb{R}^{C_{\text{out}} \times C_{\text{in}} k^2}$;
3. the convolution output is $\mathbf{W}\mathbf{P} \in \mathbb{R}^{C_{\text{out}} \times L}$, reshaped to
   $(C_{\text{out}}, H_{\text{out}}, W_{\text{out}})$.

The multiply-accumulate count is the same as for the direct computation,

$$
\text{MACs} = C_{\text{out}} \cdot C_{\text{in}} \cdot k^2 \cdot H_{\text{out}} \cdot W_{\text{out}},
$$

but the memory is not: the patch matrix duplicates every input value $k^2$ times (9x for a $3 \times 3$
kernel). PyTorch exposes im2col as `F.unfold`, and we implement it ourselves to check that we understand
the layout.

In [ ]:
def im2col(x, k, stride=1, padding=0):
    """Unfold x (C, H, W) into a matrix of flattened patches with shape (C*k*k, H_out*W_out)."""
    if padding:
        x = np.pad(x, ((0, 0), (padding, padding), (padding, padding)))
    C, H, W = x.shape
    H_out, W_out = (H - k) // stride + 1, (W - k) // stride + 1
    cols = np.zeros((C * k * k, H_out * W_out), dtype=x.dtype)
    col = 0
    for i in range(H_out):
        for j in range(W_out):
            patch = x[:, i * stride:i * stride + k, j * stride:j * stride + k]   # (C, k, k)
            cols[:, col] = patch.reshape(-1)
            col += 1
    return cols, (H_out, W_out)


def conv2d_im2col(x, w, b=None, stride=1, padding=0):
    """Convolution of x (C_in, H, W) with w (C_out, C_in, k, k) as ONE matrix multiplication."""
    C_out, C_in, k, _ = w.shape
    cols, (H_out, W_out) = im2col(x, k, stride, padding)
    out = w.reshape(C_out, -1) @ cols                 # (C_out, C_in*k*k) @ (C_in*k*k, L)
    if b is not None:
        out = out + b[:, None]
    return out.reshape(C_out, H_out, W_out)


x_np = rng.standard_normal((3, 10, 10)).astype(np.float32)
w_np = rng.standard_normal((8, 3, 3, 3)).astype(np.float32)

cols, _ = im2col(x_np, 3, stride=1, padding=1)
cols_ref = F.unfold(torch.tensor(x_np)[None], kernel_size=3, padding=1)[0]
print("patch matrix:", cols.shape, "| max |ours - F.unfold| =", f"{np.abs(cols - to_np(cols_ref)).max():.1e}")

out = conv2d_im2col(x_np, w_np, stride=1, padding=1)
ref = F.conv2d(torch.tensor(x_np)[None], torch.tensor(w_np), padding=1)[0]
print("conv via matmul:", out.shape, "| max |ours - F.conv2d| =", f"{np.abs(out - to_np(ref)).max():.1e}")
print(f"input has {x_np.size:,} values; the patch matrix has {cols.size:,} ({cols.size / x_np.size:.1f}x more memory)")

The patch matrix is nine times larger than the input, exactly the $k^2$ duplication we predicted, and the
single matrix product reproduces `F.conv2d`. Production libraries (cuDNN, oneDNN) offer several
algorithms and pick one per layer: *implicit GEMM* (im2col without materialising the patch matrix),
*Winograd* minimal filtering for $3 \times 3$ kernels (fewer multiplications), and FFT-based convolution
for large kernels. That is why `torch.backends.cudnn.benchmark = True` can speed up training: it times the
candidates for each layer shape once and caches the winner.

To put numbers on "the loop is slow", we time our reference implementation against `F.conv2d` on a
modest $64 \times 64$ image.

In [ ]:
x_np = rng.standard_normal((64, 64)).astype(np.float32)
w_np = rng.standard_normal((3, 3)).astype(np.float32)
xt, wt = torch.tensor(x_np)[None, None], torch.tensor(w_np)[None, None]

t0 = time.perf_counter(); _ = corr2d(x_np, w_np, padding=1); t_loop = time.perf_counter() - t0
F.conv2d(xt, wt, padding=1)                                  # warm-up
t0 = time.perf_counter()
for _ in range(100):
    F.conv2d(xt, wt, padding=1)
t_torch = (time.perf_counter() - t0) / 100
print(f"Python loops : {t_loop * 1e3:8.2f} ms")
print(f"F.conv2d     : {t_torch * 1e3:8.3f} ms   ({t_loop / t_torch:,.0f}x faster)")

Two to three orders of magnitude, on a tiny single-channel example; for a real layer with 64 input and
output channels the gap is far larger. From here on we use `nn.Conv2d`/`F.conv2d` exclusively, safe in the
knowledge that we have verified what they compute.

### 2.8 The `nn.Conv2d` API and shape checks

`nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True,
padding_mode='zeros')` creates the layer. A few points worth knowing:

- The weight has shape `(out_channels, in_channels // groups, kH, kW)` and is initialised with a scaled
  uniform distribution (Kaiming-uniform with `a=sqrt(5)`); for ReLU networks we will re-initialise with
  Kaiming-normal as in Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization,
  Regularization.
- `padding="same"` computes the padding for you (only for stride 1), and `padding_mode` can be
  `'reflect'` or `'replicate'` to avoid the dark border that zero padding produces.
- `groups=g` splits the input channels into $g$ groups, each convolved with its own set of kernels
  (Section 3.5: `groups=in_channels` is a depthwise convolution).
- `bias=False` is standard when a BatchNorm layer follows, because BatchNorm's own shift makes the bias
  redundant.
- The layer accepts any spatial size; only the channel count is fixed. Feeding the wrong number of
  channels is the classic error, and its message is worth learning to read.

In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
print(conv)
print("weight:", tuple(conv.weight.shape), "| bias:", tuple(conv.bias.shape),
      "| parameters:", count_params(conv), "= 16*3*3*3 + 16")

x = torch.randn(8, 3, 32, 32)
print("input", tuple(x.shape), "-> output", tuple(conv(x).shape))
print("same layer on a 64x48 image ->", tuple(conv(torch.randn(2, 3, 64, 48)).shape))

same5 = nn.Conv2d(3, 16, kernel_size=5, padding="same")
print("padding='same', k=5 ->", tuple(same5(x).shape))

grouped = nn.Conv2d(16, 32, kernel_size=3, padding=1, groups=4)
print("groups=4 weight:", tuple(grouped.weight.shape), "= (C_out, C_in/groups, k, k)")

try:
    conv(torch.randn(8, 1, 32, 32))
except RuntimeError as e:
    print("channel mismatch error:", str(e).splitlines()[0][:120])

> **Common pitfall.** `nn.Conv2d` always expects a 4-D input `(N, C, H, W)`. A single image of shape
> `(C, H, W)` must be given a batch dimension with `x.unsqueeze(0)` or `x[None]`; a grayscale image of
> shape `(H, W)` needs two. PyTorch 2.x accepts unbatched `(C, H, W)` input in many layers, but mixing the
> two conventions in one pipeline is a reliable way to produce baffling shape errors.

## 3. Pooling, strides and downsampling

### 3.1 Pooling

A **pooling** layer summarises each small window of a feature map by a single number: the maximum
(**max pooling**) or the mean (**average pooling**). With a $2 \times 2$ window and stride 2 it halves the
height and width, keeping the number of channels. Pooling has no parameters. Its purposes are to reduce
computation for the following layers, to enlarge the receptive field (it multiplies the jump by 2), and to
introduce a small amount of translation *invariance*: if a feature moves by one pixel but stays inside the
same pooling window, the pooled output is unchanged. We implement it with loops and check against PyTorch.

In [ ]:
def pool2d(x, k, stride=None, mode="max"):
    """Max or average pooling of a single-channel map x (H, W) with a k x k window."""
    stride = stride or k
    H, W = x.shape
    H_out, W_out = (H - k) // stride + 1, (W - k) // stride + 1
    y = np.zeros((H_out, W_out), dtype=x.dtype)
    for i in range(H_out):
        for j in range(W_out):
            window = x[i * stride:i * stride + k, j * stride:j * stride + k]
            y[i, j] = window.max() if mode == "max" else window.mean()
    return y


x_np = rng.standard_normal((6, 6)).astype(np.float32)
for mode, fn in [("max", F.max_pool2d), ("avg", F.avg_pool2d)]:
    ours = pool2d(x_np, 2, mode=mode)
    ref = fn(torch.tensor(x_np)[None, None], kernel_size=2)[0, 0]
    print(f"{mode}-pool 2x2: output {ours.shape}, max |ours - torch| = {np.abs(ours - to_np(ref)).max():.1e}")

# Small-shift invariance: a vertical edge at column 2 and at column 3 (same 2x2 window) pool identically
edge_a, edge_b, edge_c = np.zeros((4, 8)), np.zeros((4, 8)), np.zeros((4, 8))
edge_a[:, 2], edge_b[:, 3], edge_c[:, 4] = 1, 1, 1
print("\npooled edge at column 2:", pool2d(edge_a, 2)[0])
print("pooled edge at column 3:", pool2d(edge_b, 2)[0], "<- identical: the shift stayed inside the window")
print("pooled edge at column 4:", pool2d(edge_c, 2)[0], "<- different: the shift crossed a window boundary")

### 3.2 Strided convolutions instead of pooling

A convolution with stride 2 also halves the resolution, and unlike pooling it *learns* how to combine the
pixels it discards. Most architectures since ResNet (2015) downsample with stride-2 convolutions and use
max pooling only once, in the stem. The two options cost the same number of parameters, but a stride-2
convolution computes four times fewer output positions than a stride-1 convolution followed by pooling.

In [ ]:
x = torch.randn(1, 64, 32, 32)
conv_s1 = nn.Conv2d(64, 128, 3, stride=1, padding=1)
conv_s2 = nn.Conv2d(64, 128, 3, stride=2, padding=1)
y_pool = F.max_pool2d(conv_s1(x), 2)
y_stride = conv_s2(x)
macs_s1 = 32 * 32 * 128 * 64 * 9
macs_s2 = 16 * 16 * 128 * 64 * 9
print(f"conv(s=1) + maxpool : output {tuple(y_pool.shape)}, {count_params(conv_s1):,} params, {macs_s1 / 1e6:6.1f} M MACs")
print(f"conv(s=2)           : output {tuple(y_stride.shape)}, {count_params(conv_s2):,} params, {macs_s2 / 1e6:6.1f} M MACs")

### 3.3 Global average pooling

Older CNNs (LeNet, AlexNet, VGG) flatten the last feature map and feed it to large fully connected layers,
which contain most of the parameters (VGG-16: 124 million of its 138 million) and tie the network to one
input size. **Global average pooling** (GAP) instead averages each channel over all spatial positions,
turning a $(N, C, H, W)$ map into a $(N, C)$ vector regardless of $H$ and $W$. A single linear layer then
maps $C$ features to the classes. This was introduced with Network-in-Network (Lin et al., 2014) and
adopted by GoogLeNet and ResNet; it removes almost all classifier parameters and makes the network
input-size agnostic. In PyTorch it is `nn.AdaptiveAvgPool2d(1)`, which adapts the window so that the
output is $1 \times 1$.

In [ ]:
gap = nn.AdaptiveAvgPool2d(1)
for size in [32, 48, 64]:
    feat = torch.randn(2, 128, size, size)
    print(f"feature map {str(tuple(feat.shape)):18s} -> GAP {tuple(gap(feat).shape)} -> flatten {tuple(gap(feat).flatten(1).shape)}")

### 3.4 1x1 convolutions

A $1 \times 1$ convolution looks pointless at first: it has no spatial extent. But with many channels it is
a fully connected layer *applied independently at every pixel*, mixing the $C_{\text{in}}$ channel values
at that position into $C_{\text{out}}$ new ones. It is used to change the number of channels cheaply
(a "bottleneck" that reduces 256 channels to 64 before an expensive $3 \times 3$ convolution, as in the
ResNet-50 block), to add non-linearity without spatial mixing, and as the **projection shortcut** of
residual blocks (Section 6.3). We verify the equivalence with `torch.einsum` and with an `nn.Linear`
applied to a channels-last view.

In [ ]:
torch.manual_seed(SEED)
conv1x1 = nn.Conv2d(64, 16, kernel_size=1)
x = torch.randn(2, 64, 8, 8)
y_conv = conv1x1(x)

W1 = conv1x1.weight[:, :, 0, 0]                                    # (16, 64)
y_einsum = torch.einsum("oc,nchw->nohw", W1, x) + conv1x1.bias[None, :, None, None]
print("max |conv1x1 - einsum|    :", f"{(y_conv - y_einsum).abs().max().item():.1e}")

lin = nn.Linear(64, 16)
with torch.no_grad():
    lin.weight.copy_(W1); lin.bias.copy_(conv1x1.bias)
y_linear = lin(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)          # NCHW -> NHWC -> Linear -> NCHW
print("max |conv1x1 - nn.Linear| :", f"{(y_conv - y_linear).abs().max().item():.1e}")

### 3.5 Depthwise-separable convolutions

A standard convolution mixes space and channels at once, and its cost
$C_{\text{out}} C_{\text{in}} k^2 H W$ grows with the *product* of the channel counts. A
**depthwise-separable convolution** (Howard et al., 2017, MobileNets) factorises it into two cheaper steps:

1. a **depthwise** convolution (`groups=C_in`): one $k \times k$ kernel *per channel*, no mixing across
   channels, cost $C_{\text{in}} k^2 H W$;
2. a **pointwise** $1 \times 1$ convolution that mixes channels, cost $C_{\text{out}} C_{\text{in}} H W$.

The ratio of the separable cost to the standard cost is

$$
\frac{C_{\text{in}} k^2 + C_{\text{out}} C_{\text{in}}}{C_{\text{out}} C_{\text{in}} k^2}
= \frac{1}{C_{\text{out}}} + \frac{1}{k^2},
$$

which for $k = 3$ and any reasonably large $C_{\text{out}}$ is about $1/9$: an 8-9x saving in both
parameters and multiply-accumulates, for a modest loss in accuracy. This is the workhorse of mobile
architectures. A small helper computes parameters and MACs from an `nn.Conv2d` and the input size.

In [ ]:
def conv_cost(conv, H, W):
    """(parameters, multiply-accumulates) of an nn.Conv2d applied to an (H, W) input."""
    kh, kw = conv.kernel_size
    H_out = conv_out_size(H, kh, conv.stride[0], conv.padding[0], conv.dilation[0])
    W_out = conv_out_size(W, kw, conv.stride[1], conv.padding[1], conv.dilation[1])
    macs = H_out * W_out * conv.out_channels * (conv.in_channels // conv.groups) * kh * kw
    return count_params(conv), macs


C_in, C_out, size = 128, 128, 32
standard = nn.Conv2d(C_in, C_out, 3, padding=1)
depthwise = nn.Conv2d(C_in, C_in, 3, padding=1, groups=C_in)
pointwise = nn.Conv2d(C_in, C_out, 1)
p_std, m_std = conv_cost(standard, size, size)
p_dw, m_dw = conv_cost(depthwise, size, size)
p_pw, m_pw = conv_cost(pointwise, size, size)

print(f"{'layer (128 -> 128 channels, 32x32)':36s} {'params':>10s} {'MACs':>14s}")
for name, p, m in [("standard 3x3", p_std, m_std), ("depthwise 3x3", p_dw, m_dw),
                   ("pointwise 1x1", p_pw, m_pw), ("separable = depthwise + pointwise", p_dw + p_pw, m_dw + m_pw)]:
    print(f"{name:36s} {p:10,d} {m:14,d}")
print(f"\nreduction: params {p_std / (p_dw + p_pw):.1f}x, MACs {m_std / (m_dw + m_pw):.1f}x "
      f"(theory 1/(1/C_out + 1/k^2) = {1 / (1 / C_out + 1 / 9):.1f}x)")

### 3.6 Transposed convolutions (brief)

So far every operation kept or reduced the resolution. Generative models (Notebook 11 · Autoencoders and
Variational Autoencoders, Notebook 12 · GANs and Diffusion Models) and segmentation networks (Notebook 07 ·
Transfer Learning and Modern Vision) need the opposite: to go from a small, deep feature map back to a large image. The **transposed
convolution** `nn.ConvTranspose2d` does this. Its name comes from the im2col picture: if a convolution with
stride $s$ is the linear map $y = \mathbf{A} x$, the transposed convolution is $\mathbf{A}^\top$, which maps
from the output space back to the input space, and therefore *upsamples* by the factor $s$. The output
size is

$$
n_{\text{out}} = (n - 1)\, s - 2p + d(k - 1) + \text{output\_padding} + 1 ,
$$

where `output_padding` resolves the ambiguity created by the floor in the forward formula. A common
configuration for doubling the resolution is $k = 4$, $s = 2$, $p = 1$. The cell also verifies the
"transpose" claim: the gradient of a convolution with respect to its input (a vector-Jacobian product,
Notebook 03 · Tensors and Autograd) is exactly a transposed convolution with the same kernel.

In [ ]:
up = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
x = torch.randn(1, 64, 8, 8)
print("ConvTranspose2d(k=4, s=2, p=1):", tuple(x.shape), "->", tuple(up(x).shape))


def conv_transpose_out_size(n, k, stride=1, padding=0, output_padding=0, dilation=1):
    return (n - 1) * stride - 2 * padding + dilation * (k - 1) + output_padding + 1


print("formula:", conv_transpose_out_size(8, 4, stride=2, padding=1))

# The transposed convolution is the adjoint (vector-Jacobian product) of the convolution.
torch.manual_seed(SEED)
w = torch.randn(1, 1, 3, 3)
x = torch.randn(1, 1, 7, 7, requires_grad=True)
y = F.conv2d(x, w, stride=2)                       # (1, 1, 3, 3)
v = torch.randn_like(y)
(y * v).sum().backward()                           # x.grad = J^T v
adjoint = F.conv_transpose2d(v, w, stride=2)       # should be the same vector
print("conv:", tuple(x.shape), "->", tuple(y.shape), "| conv_transpose:", tuple(v.shape), "->", tuple(adjoint.shape))
print("max |x.grad - conv_transpose2d(v)| =", f"{(x.grad - adjoint).abs().max().item():.1e}")

Transposed convolutions with $k$ not divisible by $s$ produce **checkerboard artifacts** (some output
pixels receive contributions from more kernel taps than others), which is why many modern decoders use
`nn.Upsample(scale_factor=2)` (nearest or bilinear interpolation) followed by an ordinary convolution
instead. We return to both options in Notebook 07 · Transfer Learning and Modern Vision (U-Net) and
Notebook 12 · GANs and Diffusion Models (DCGAN).

## 4. LeNet-5 on MNIST

### 4.1 The architecture

LeNet-5 (LeCun, Bottou, Bengio and Haffner, 1998) is the archetype of the convolutional network and was
used commercially to read cheques and postcodes. Its structure is still the template for every CNN:
two blocks of *convolution → non-linearity → pooling* that build a hierarchy of local features, followed
by fully connected layers that combine them into a decision.

| layer | operation | output shape (C, H, W) | parameters |
|---|---|---|---|
| input | grayscale digit | (1, 28, 28) | – |
| C1 | conv 5x5, 6 filters, padding 2 | (6, 28, 28) | 6·(1·25) + 6 = 156 |
| S2 | pool 2x2 | (6, 14, 14) | 0 |
| C3 | conv 5x5, 16 filters | (16, 10, 10) | 16·(6·25) + 16 = 2,416 |
| S4 | pool 2x2 | (16, 5, 5) | 0 |
| C5/F5 | linear 400 → 120 | (120,) | 48,120 |
| F6 | linear 120 → 84 | (84,) | 10,164 |
| output | linear 84 → 10 | (10,) | 850 |

The original used a scaled tanh activation, average pooling ("subsampling" layers with a trainable gain
and bias), a hand-designed sparse connection table between S2 and C3 (each C3 map saw only some of the S2
maps, so C3 had 1,516 parameters rather than the 2,416 of our fully connected version) and a radial-basis
output layer; we use the modern equivalents (ReLU, max pooling, full connections, a linear layer with
softmax cross-entropy), which train faster and slightly better. Notice where the parameters are: 61,706
in total, of which 59,134 (96 %) sit in the three linear layers. The convolutional part is tiny but does the hard work of turning pixels into features.

In [ ]:
class LeNet5(nn.Module):
    """LeNet-5 (LeCun et al., 1998) with ReLU and max pooling, for 1x28x28 inputs."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, padding=2),   # (1, 28, 28) -> (6, 28, 28)
            nn.ReLU(),
            nn.MaxPool2d(2),                             # -> (6, 14, 14)
            nn.Conv2d(6, 16, kernel_size=5),             # -> (16, 10, 10)
            nn.ReLU(),
            nn.MaxPool2d(2),                             # -> (16, 5, 5)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                # -> (400,)
            nn.Linear(16 * 5 * 5, 120), nn.ReLU(),
            nn.Linear(120, 84), nn.ReLU(),
            nn.Linear(84, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def trace_shapes(model, input_shape):
    """Run one zero input through the model and print every leaf module's output shape."""
    handles = []

    def hook(module, inputs, output):
        print(f"{module.__class__.__name__:14s} {count_params(module):>8,d} params -> {tuple(output.shape)}")

    for m in model.modules():
        if len(list(m.children())) == 0:                 # leaf modules only
            handles.append(m.register_forward_hook(hook))
    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, *input_shape))
    for h in handles:
        h.remove()
    print(f"total trainable parameters: {count_params(model):,}")


lenet = LeNet5()
trace_shapes(lenet, (1, 28, 28))

The forward hooks confirm the shape table: two convolutions and two poolings take $(1, 28, 28)$ to
$(16, 5, 5) = 400$ numbers, which the classifier maps to 10 logits. (`trace_shapes` is a miniature version
of `torchinfo.summary`, which Notebook 05 · The PyTorch Workflow and Best Practices introduced.)

### 4.2 Data and the training helpers

We load MNIST through `torchvision.datasets` with a `transforms.v2` pipeline that converts the PIL image
to a `float32` tensor in $[0, 1]$ and standardises it with the dataset's mean and standard deviation
(0.1307 and 0.3081, values every MNIST tutorial hard-codes; Section 5.2 shows how to compute them). Under
`FAST_RUN` we train on the first 5,000 digits and test on 2,000; otherwise on the full 60,000 / 10,000.

Then we define the three functions that every experiment in this notebook reuses: `train_one_epoch`,
`evaluate` and `fit`. They follow the pattern of Notebook 05 · The PyTorch Workflow and Best Practices,
with two additions. First, `fit` records
the loss of *every iteration*, not only per epoch, because the degradation experiment in Section 6 needs
the fine-grained curve. Second, on a GPU the forward pass runs under `torch.amp.autocast` with a
`GradScaler` (mixed precision, explained in Notebook 14 · Efficient Training and Scaling); on a CPU these
calls are disabled and cost nothing. The optimiser is SGD with Nesterov momentum and weight decay, the
standard choice for CNNs, with a cosine learning-rate schedule stepped every iteration.

In [ ]:
mnist_tf = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
                       v2.Normalize(mean=(0.1307,), std=(0.3081,))])
mnist_train = datasets.MNIST(str(DATA_DIR), train=True, download=True, transform=mnist_tf)
mnist_test = datasets.MNIST(str(DATA_DIR), train=False, download=True, transform=mnist_tf)

N_TRAIN_MNIST = 5_000 if FAST_RUN else len(mnist_train)
N_TEST_MNIST = 2_000 if FAST_RUN else len(mnist_test)
PIN = device.type == "cuda"
mnist_train_loader = DataLoader(Subset(mnist_train, range(N_TRAIN_MNIST)), batch_size=128,
                                shuffle=True, num_workers=0, pin_memory=PIN)
mnist_test_loader = DataLoader(Subset(mnist_test, range(N_TEST_MNIST)), batch_size=512,
                               shuffle=False, num_workers=0, pin_memory=PIN)

x0, y0 = mnist_train[0]
print(f"one sample: image {tuple(x0.shape)} {x0.dtype}, label {y0}; "
      f"train {N_TRAIN_MNIST:,} / test {N_TEST_MNIST:,} images")
xb, yb = next(iter(mnist_train_loader))
print(f"one mini-batch: images {tuple(xb.shape)}, labels {tuple(yb.shape)}, "
      f"pixel range [{xb.min():.2f}, {xb.max():.2f}]")

The pixel range is no longer $[0, 1]$ but roughly $[-0.42, 2.8]$: the background (value 0 before
normalisation) maps to $-0.1307 / 0.3081 \approx -0.42$ and the brightest stroke pixels to
$(1 - 0.1307) / 0.3081 \approx 2.8$. Now the training helpers. `train_one_epoch` and `evaluate` are the
inner loops; `fit` wires them to an optimiser, a schedule and a `GradScaler` and returns a history
dictionary that `plot_history` draws.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scheduler=None, scaler=None):
    """One pass over `loader`. Returns (mean loss, accuracy, list of per-iteration losses)."""
    model.train()
    total_loss, correct, n, iter_losses = 0.0, 0, 0, []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        iter_losses.append(loss.item())
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n, iter_losses


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Mean loss and accuracy of `model` on `loader` (no gradients, eval mode)."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n


def fit(model, train_loader, test_loader, epochs, lr=0.05, weight_decay=5e-4, momentum=0.9,
        criterion=None, scheduler_fn="cosine", verbose=True):
    """Train `model` with SGD + Nesterov momentum and (by default) a per-iteration cosine schedule.

    scheduler_fn: "cosine", None, or a callable (optimizer, total_steps) -> scheduler.
    Returns a history dict with per-epoch metrics and the per-iteration training loss.
    """
    model.to(device)
    criterion = criterion if criterion is not None else nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum,
                                weight_decay=weight_decay, nesterov=True)
    total_steps = epochs * len(train_loader)
    if scheduler_fn == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
    elif scheduler_fn is None:
        scheduler = None
    else:
        scheduler = scheduler_fn(optimizer, total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    hist = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": [], "iter_loss": [], "epoch_time": []}
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc, iters = train_one_epoch(model, train_loader, optimizer, criterion, scheduler, scaler)
        te_loss, te_acc = evaluate(model, test_loader, criterion)
        dt = time.time() - t0
        hist["train_loss"].append(tr_loss); hist["train_acc"].append(tr_acc)
        hist["test_loss"].append(te_loss); hist["test_acc"].append(te_acc)
        hist["iter_loss"] += iters; hist["epoch_time"].append(dt)
        if verbose:
            print(f"epoch {epoch:2d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
                  f"test loss {te_loss:.4f} acc {te_acc:.3f} | {dt:5.1f}s")
    return hist


def plot_history(histories, title=None):
    """Loss and accuracy curves for a dict {name: history}."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, h in histories.items():
        ep = range(1, len(h["train_loss"]) + 1)
        axes[0].plot(ep, h["train_loss"], "o--", label=f"{name} (train)")
        axes[0].plot(ep, h["test_loss"], "o-", label=f"{name} (test)")
        axes[1].plot(ep, [100 * a for a in h["test_acc"]], "o-", label=name)
    axes[0].set(xlabel="epoch", ylabel="cross-entropy", title="loss"); axes[0].legend()
    axes[1].set(xlabel="epoch", ylabel="test accuracy (%)", title="accuracy"); axes[1].legend()
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()


print("training helpers defined")

### 4.3 Training LeNet-5

One epoch on the 5,000-digit subset (`FAST_RUN`) already gives around 90 %; three epochs on the full
training set give about 99 %, which is where LeNet-style networks were in the late 1990s (the original
paper reports 0.95 % test error for LeNet-5 without augmentation).

In [ ]:
EPOCHS_MNIST = 1 if FAST_RUN else 3
torch.manual_seed(SEED)
lenet = LeNet5()
print(f"LeNet-5: {count_params(lenet):,} parameters, {EPOCHS_MNIST} epoch(s) on {N_TRAIN_MNIST:,} digits")
hist_lenet = fit(lenet, mnist_train_loader, mnist_test_loader, epochs=EPOCHS_MNIST, lr=0.05)

For comparison, the 784-256-128-10 MLP of Notebook 02 · Neural Networks from Scratch (NumPy) reached
about 97 % with 235,000 parameters;
LeNet-5 reaches about 99 % with a quarter of that. The gain comes entirely from the inductive biases of
Section 1.3: the convolutional layers do not need to learn separately what a stroke looks like at every
position of the image.

### 4.4 What did the first layer learn?

The first-layer kernels of LeNet are $5 \times 5$ single-channel patterns, so we can draw them directly.
We also feed one test digit through the network and display the six feature maps after the first
convolution and ReLU, and the sixteen after the second, using slices of `lenet.features`.

In [ ]:
lenet.eval()
w1 = to_np(lenet.features[0].weight)[:, 0]                       # (6, 5, 5)

fig, axes = plt.subplots(1, 6, figsize=(11, 2.2))
for i, ax in enumerate(axes):
    ax.imshow(w1[i], cmap="RdBu_r", vmin=-abs(w1).max(), vmax=abs(w1).max())
    ax.set_title(f"kernel {i}"); ax.axis("off")
plt.suptitle("LeNet-5 first-layer kernels (red = positive, blue = negative)"); plt.tight_layout(); plt.show()

x_digit, y_digit = mnist_test[7]
with torch.no_grad():
    fm1 = lenet.features[:2](x_digit[None].to(device))[0]           # after conv1 + ReLU: (6, 28, 28)
    fm2 = lenet.features[:5](x_digit[None].to(device))[0]           # after conv2 + ReLU: (16, 10, 10)

fig, axes = plt.subplots(3, 8, figsize=(13, 5.2))
axes = list(axes.flat)
axes[0].imshow(to_np(x_digit[0]), cmap="gray"); axes[0].set_title(f"digit, label {y_digit}")
for i in range(6):
    axes[1 + i].imshow(to_np(fm1[i]), cmap="viridis"); axes[1 + i].set_title(f"conv1 map {i}")
for i in range(16):
    axes[8 + i].imshow(to_np(fm2[i]), cmap="viridis"); axes[8 + i].set_title(f"conv2 map {i}", fontsize=8)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

The first-layer kernels are small oriented edge and stroke detectors, positive on one side and negative on
the other, exactly the kind of pattern we designed by hand in Section 2.5. The first feature maps are
therefore edge maps of the digit at different orientations. The second-layer maps are coarser
($10 \times 10$) and sparser: each one responds to a *combination* of first-layer edges (a corner, the end
of a stroke, a curve), which is the beginning of the feature hierarchy that deep CNNs extend over dozens
of layers. Section 8 repeats this analysis on the deeper CIFAR-10 networks.

> **Try it.** Change the index of the test digit (`mnist_test[7]`) and watch how the same kernels respond
> to a different shape. The kernels are fixed; only their responses change.

## 5. CIFAR-10 and a VGG-style network

### 5.1 The dataset

MNIST is solved; CIFAR-10 (Krizhevsky, 2009) is the smallest dataset on which architecture choices still
matter. It has 60,000 colour images of $32 \times 32$ pixels in ten classes (airplane, automobile, bird,
cat, deer, dog, frog, horse, ship, truck), 50,000 for training and 10,000 for testing. The images are
tiny, blurry and varied: objects appear at different scales, poses and backgrounds. A linear classifier
reaches about 40 %, a well-tuned MLP about 55-60 %, a small CNN 80 %+, a ResNet 93-95 %, and the best
published numbers (with heavy augmentation or pretraining) exceed 99 %.

`torchvision.datasets.CIFAR10` downloads the data into `DATA_DIR` and keeps all images in memory as a
single NumPy array of shape `(N, 32, 32, 3)`. Instead of going through PIL for every sample, we convert
that array once into a channels-first `uint8` tensor and write a tiny `Dataset` that applies a
`transforms.v2` pipeline to tensors. With `num_workers=0` (the safe setting in notebooks, especially on
Windows) the data pipeline runs in the main process, so keeping it cheap matters. If the download fails
(no internet), the cell falls back to a synthetic CIFAR-like set of coloured shapes with the same layout,
so every later cell still runs; its accuracies are not comparable to real CIFAR-10 numbers.

In [ ]:
def _synthetic_cifar_like(n, seed):
    """Offline stand-in for CIFAR-10: 32x32 RGB images of ten colour-and-shape classes (random position,
    size, background and pixel noise). Same array layout as torchvision's CIFAR-10, so every cell below
    runs unchanged. Accuracies on it are NOT comparable to real CIFAR-10 results."""
    rng = np.random.default_rng(seed)
    labels = rng.integers(0, 10, size=n)
    colours = np.array([[210, 60, 60], [60, 190, 60], [60, 80, 220], [220, 200, 50], [200, 60, 200],
                        [60, 200, 210], [235, 140, 40], [130, 50, 220], [60, 220, 150], [170, 170, 170]], dtype=np.float32)
    yy, xx = np.mgrid[0:32, 0:32]
    images = np.empty((n, 32, 32, 3), dtype=np.uint8)
    for i in range(n):
        c = int(labels[i])
        img = np.ones((32, 32, 3), np.float32) * (rng.uniform(40, 140, size=3) + rng.normal(0, 4, size=3))
        cx, cy, r = rng.uniform(11, 21), rng.uniform(11, 21), rng.uniform(5, 10)
        dx, dy = xx - cx, yy - cy
        shape = c % 5                                       # 0 square, 1 disc, 2 triangle, 3 ring, 4 stripes
        if shape == 0:
            mask = (np.abs(dx) < r) & (np.abs(dy) < r)
        elif shape == 1:
            mask = dx ** 2 + dy ** 2 < r ** 2
        elif shape == 2:
            mask = (dy > -r) & (np.abs(dx) < (dy + r) * 0.6)
        elif shape == 3:
            d2 = dx ** 2 + dy ** 2
            mask = (d2 < r ** 2) & (d2 > (0.55 * r) ** 2)
        else:
            mask = (np.abs(dx) < r) & (np.abs(dy) < r) & (((yy - int(cy)) // 3) % 2 == 0)
        img[mask] = colours[c] * rng.uniform(0.85, 1.15, size=3)
        img += rng.normal(0, 10, img.shape)
        images[i] = np.clip(img, 0, 255).astype(np.uint8)
    return images, labels.tolist()


def load_cifar10_or_fallback(data_dir, n_train=10_000, n_test=2_000, seed=SEED):
    """Download CIFAR-10 if possible; when offline, return a synthetic CIFAR-like dataset so that the
    notebook still runs end to end. Returns (train, test, is_synthetic); both splits expose
    .data (N, 32, 32, 3) uint8, .targets (list of int) and .classes like torchvision's CIFAR10."""
    try:
        train = datasets.CIFAR10(str(data_dir), train=True, download=True)
        test = datasets.CIFAR10(str(data_dir), train=False, download=True)
        return train, test, False
    except Exception as e:                                   # no internet, blocked host, corrupt download
        from types import SimpleNamespace
        print(f"[offline fallback] CIFAR-10 could not be downloaded ({type(e).__name__}: {str(e)[:80]}).\n"
              f"  Using a SYNTHETIC CIFAR-like dataset of coloured shapes ({n_train:,} train / {n_test:,} test images)\n"
              f"  so that the rest of the notebook runs. Numbers below are NOT comparable to real CIFAR-10 results;\n"
              f"  re-run this cell with internet access (or place cifar-10-python.tar.gz in {data_dir}) for the real data.")
        classes = ["red square", "green disc", "blue triangle", "yellow ring", "magenta stripes",
                   "cyan square", "orange disc", "violet triangle", "mint ring", "grey stripes"]
        xtr, ytr = _synthetic_cifar_like(n_train, seed)
        xte, yte = _synthetic_cifar_like(n_test, seed + 1)
        return (SimpleNamespace(data=xtr, targets=ytr, classes=classes),
                SimpleNamespace(data=xte, targets=yte, classes=classes), True)


# Offline, the synthetic stand-in only needs as many images as the experiments below use.
cifar_train_raw, cifar_test_raw, CIFAR_IS_SYNTHETIC = load_cifar10_or_fallback(
    DATA_DIR, n_train=5_000 if FAST_RUN else 50_000, n_test=2_000 if FAST_RUN else 10_000)
CLASSES = cifar_train_raw.classes

cifar_train_x = torch.from_numpy(cifar_train_raw.data).permute(0, 3, 1, 2).contiguous()   # uint8 (50000, 3, 32, 32)
cifar_train_y = torch.tensor(cifar_train_raw.targets, dtype=torch.long)
cifar_test_x = torch.from_numpy(cifar_test_raw.data).permute(0, 3, 1, 2).contiguous()
cifar_test_y = torch.tensor(cifar_test_raw.targets, dtype=torch.long)

print("train:", tuple(cifar_train_x.shape), cifar_train_x.dtype, "| test:", tuple(cifar_test_x.shape))
print("classes:", CLASSES)
print("images per class (train):", torch.bincount(cifar_train_y).tolist())

fig, axes = plt.subplots(2, 10, figsize=(14, 3.4))
for c in range(10):
    idx = torch.nonzero(cifar_train_y == c).flatten()[:2]
    for r in range(2):
        axes[r, c].imshow(to_np(cifar_train_x[idx[r]].permute(1, 2, 0)))
        axes[r, c].axis("off")
    axes[0, c].set_title(CLASSES[c], fontsize=9)
plt.tight_layout(); plt.show()

### 5.2 Normalisation statistics

Networks train best when the inputs have roughly zero mean and unit variance per channel (Notebook 04 ·
Training Deep Networks: Optimization, Initialization, Normalization, Regularization discussed why in the
context of initialisation). We compute the per-channel mean and standard deviation
of the training images once, in chunks to keep the memory footprint small, and use them for both the
training and the test pipeline. Computing statistics on the *training* set only is a rule, not a detail:
the test set must not influence any decision, including preprocessing.

In [ ]:
def channel_stats(images_uint8, chunk=5_000):
    """Per-channel mean and std of a uint8 image tensor (N, C, H, W), computed in chunks."""
    C = images_uint8.shape[1]
    s = torch.zeros(C, dtype=torch.float64)
    s2 = torch.zeros(C, dtype=torch.float64)
    n = 0
    for part in images_uint8.split(chunk):
        x = part.to(torch.float64) / 255.0
        s += x.sum(dim=(0, 2, 3))
        s2 += (x ** 2).sum(dim=(0, 2, 3))
        n += x.shape[0] * x.shape[2] * x.shape[3]
    mean = s / n
    std = (s2 / n - mean ** 2).sqrt()
    return mean.float(), std.float()


cifar_mean, cifar_std = channel_stats(cifar_train_x)
CIFAR_MEAN, CIFAR_STD = tuple(cifar_mean.tolist()), tuple(cifar_std.tolist())
print("per-channel mean (R, G, B):", [f"{m:.4f}" for m in CIFAR_MEAN])
print("per-channel std  (R, G, B):", [f"{s:.4f}" for s in CIFAR_STD])

These are the numbers (0.491, 0.482, 0.447) and (0.247, 0.243, 0.262) that CIFAR-10 code hard-codes
everywhere; now you know where they come from.

### 5.3 Data augmentation with `transforms.v2`

With 50,000 training images and networks of millions of parameters, overfitting is guaranteed unless we
regularise. The most effective regulariser for images is **data augmentation** (Notebook 04, Section 6):
random, label-preserving transformations that show the network a slightly different image every epoch.
For CIFAR-10 the standard recipe since the ResNet paper is

- **random crop with padding**: pad the image by 4 pixels on each side, then crop a random $32 \times 32$
  window (a random translation by up to 4 pixels);
- **random horizontal flip** with probability 0.5 (a mirrored cat is still a cat; a mirrored digit would
  not be, so this augmentation is dataset-specific).

`ColorJitter` (random brightness, contrast, saturation and hue changes) is another common choice; it helps
a little on CIFAR-10 but costs noticeable CPU time per image, so we show it in the gallery and leave it
out of the training pipeline. Augmentation is applied only to the training data; the test pipeline just
converts to `float32` and normalises. `transforms.v2` operates directly on `uint8` tensors of shape
`(C, H, W)`, which is what our `Dataset` stores.

In [ ]:
train_tf = v2.Compose([
    v2.RandomCrop(32, padding=4),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])
eval_tf = v2.Compose([
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])
jitter_tf = v2.Compose([v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.05),
                        v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD)])


def denorm(x):
    """Undo Normalize for display: normalised (3, H, W) tensor -> (H, W, 3) array in [0, 1]."""
    mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(3, 1, 1)
    return to_np((x.detach().cpu().float() * std + mean).clamp(0, 1).permute(1, 2, 0))


torch.manual_seed(SEED)
sample = cifar_train_x[4]
fig, axes = plt.subplots(2, 6, figsize=(12, 4.2))
axes[0, 0].imshow(to_np(sample.permute(1, 2, 0))); axes[0, 0].set_title(f"original ({CLASSES[cifar_train_y[4]]})")
for j in range(1, 6):
    axes[0, j].imshow(denorm(train_tf(sample))); axes[0, j].set_title("crop + flip")
axes[1, 0].imshow(denorm(eval_tf(sample))); axes[1, 0].set_title("eval pipeline")
for j in range(1, 6):
    axes[1, j].imshow(denorm(jitter_tf(sample))); axes[1, j].set_title("ColorJitter")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()

The crop-and-flip samples are all slightly shifted or mirrored versions of the same image (its class is
in the title of the first panel); the black border comes from the zero padding when the random window
extends past the image. The evaluation pipeline
reproduces the original exactly (normalisation is undone by `denorm`).

Now the `Dataset` and a loader factory. The factory takes the number of training and test images so that
the `FAST_RUN` budgets, the degradation experiment and the exercises can all request the subsets they need.
The training subset is a random (seeded) sample so that the classes stay balanced.

In [ ]:
class TensorImageDataset(Dataset):
    """Images stored as one uint8 tensor (N, C, H, W) plus integer labels, with a per-sample transform."""

    def __init__(self, images, labels, transform=None):
        self.images, self.labels, self.transform = images, labels, transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        x = self.images[i]
        if self.transform is not None:
            x = self.transform(x)
        return x, self.labels[i]


def make_cifar_loaders(n_train, n_test, batch_size=128, transform=None):
    """DataLoaders over a seeded random subset of CIFAR-10 train and the first n_test test images."""
    transform = transform if transform is not None else train_tf
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(len(cifar_train_y), generator=g)[:n_train]
    train_ds = TensorImageDataset(cifar_train_x[idx], cifar_train_y[idx], transform)
    test_ds = TensorImageDataset(cifar_test_x[:n_test], cifar_test_y[:n_test], eval_tf)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0,
                              pin_memory=PIN, drop_last=True)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0, pin_memory=PIN)
    return train_loader, test_loader


N_TRAIN = 5_000 if FAST_RUN else 50_000
N_TEST = 2_000 if FAST_RUN else 10_000
EPOCHS_CIFAR = 1 if FAST_RUN else 12
train_loader, test_loader = make_cifar_loaders(N_TRAIN, N_TEST, batch_size=128)

xb, yb = next(iter(train_loader))
print(f"train {N_TRAIN:,} images in {len(train_loader)} batches | test {N_TEST:,} images | {EPOCHS_CIFAR} epoch(s)")
print(f"batch: images {tuple(xb.shape)} {xb.dtype}, labels {tuple(yb.shape)} {yb.dtype}; "
      f"mean {xb.mean():.3f}, std {xb.std():.3f}")

### 5.4 A VGG-style network with BatchNorm

VGG (Simonyan and Zisserman, 2015) made one point with great clarity: use only $3 \times 3$ convolutions
and go deep. Two stacked $3 \times 3$ layers have the receptive field of one $5 \times 5$ layer with
$2 \cdot 9 = 18$ instead of 25 weights per channel pair *and* an extra non-linearity in between; three
stacked replace a $7 \times 7$ with 27 instead of 49. The network is organised in **stages**: a few
convolutions at a fixed resolution, then a max pooling that halves the resolution while the next stage
doubles the channels. This "halve the resolution, double the width" rule keeps the cost per stage roughly
constant and is still followed by ResNets and ConvNeXt.

Our CIFAR-sized version has three stages of two convolutions each, every convolution followed by
**BatchNorm** and ReLU (BatchNorm did not exist when VGG was published; adding it lets us use a much
larger learning rate and removes the need for the careful staged initialisation of the original paper),
and a compact VGG-like classifier: flatten, one hidden linear layer with dropout, output layer. The width
of the first stage is 32 under `FAST_RUN` and 64 in the full run; the configuration list uses the string
`"M"` for a pooling layer, exactly as in torchvision's `vgg.py`.

In [ ]:
def conv_bn_relu(in_ch, out_ch, stride=1):
    """3x3 convolution (no bias) -> BatchNorm -> ReLU."""
    return nn.Sequential(nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False),
                         nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))


def init_weights(m):
    """Kaiming-normal initialisation for convolutions (fan_out, as in torchvision's ResNet)."""
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.ones_(m.weight); nn.init.zeros_(m.bias)


class VGGStyle(nn.Module):
    """A VGG-like CNN: stages of conv-BN-ReLU separated by 'M' (max pooling), then an MLP head."""

    def __init__(self, cfg, num_classes=10, in_ch=3, in_size=32, hidden=256, dropout=0.3):
        super().__init__()
        layers, c = [], in_ch
        for v in cfg:
            if v == "M":
                layers.append(nn.MaxPool2d(2))
            else:
                layers.append(conv_bn_relu(c, v))
                c = v
        self.features = nn.Sequential(*layers)
        spatial = in_size // 2 ** cfg.count("M")
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(c * spatial * spatial, hidden), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )
        self.apply(init_weights)

    def forward(self, x):
        return self.classifier(self.features(x))


VGG_WIDTH = 32 if FAST_RUN else 64
vgg_cfg = [VGG_WIDTH, VGG_WIDTH, "M", 2 * VGG_WIDTH, 2 * VGG_WIDTH, "M", 4 * VGG_WIDTH, 4 * VGG_WIDTH, "M"]
torch.manual_seed(SEED)
vgg = VGGStyle(vgg_cfg)
print("configuration:", vgg_cfg)
trace_shapes(vgg, (3, 32, 32))

Follow the shapes: $(3, 32, 32) \to (w, 32, 32) \to (2w, 16, 16) \to (4w, 8, 8) \to (4w, 4, 4)$, then a
vector of $4w \cdot 16$ numbers. In the full configuration ($w = 64$) the six convolutions hold about
1.1 million parameters and the first linear layer another 1.0 million; in VGG-16 proper, whose classifier
alone has 124 million parameters, this imbalance was much worse.

### 5.5 Training

We train with SGD (Nesterov momentum 0.9, weight decay $5 \times 10^{-4}$), learning rate 0.05 with a
per-iteration cosine schedule, batch size 128. Under `FAST_RUN` this is one epoch on 5,000 images, which
takes well under a minute on a CPU and reaches roughly 40-50 %; the full run (12 epochs on 50,000 images)
takes a few minutes on a GPU and should land above 85 %.

In [ ]:
torch.manual_seed(SEED)
vgg = VGGStyle(vgg_cfg)
print(f"VGG-style ({count_params(vgg):,} parameters), {EPOCHS_CIFAR} epoch(s) on {N_TRAIN:,} images")
hist_vgg = fit(vgg, train_loader, test_loader, epochs=EPOCHS_CIFAR, lr=0.05)
plot_history({"VGG-style": hist_vgg})

In the full run the training loss keeps falling while the test accuracy climbs steadily and then
flattens as the cosine schedule brings the learning rate to zero; the last two or three epochs, at a very
small learning rate, typically add 2-3 points. The gap between training accuracy (with augmentation and
dropout active) and test accuracy stays small, which tells us the regularisation is doing its job. Under
`FAST_RUN` you see a single point per curve: a sanity check, not a result.

### 5.6 Which classes are hard?

Overall accuracy hides structure. A per-class breakdown and a confusion matrix show *which* mistakes the
network makes, and on CIFAR-10 the answer is always the same: cats and dogs are confused with each other,
and so are automobiles and trucks, while frogs and ships are easy.

In [ ]:
@torch.no_grad()
def predict_all(model, loader):
    """Predicted and true labels for a whole loader, as CPU tensors."""
    model.eval()
    preds, targets = [], []
    for x, y in loader:
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(x.to(device, non_blocking=True))
        preds.append(logits.argmax(dim=1).cpu()); targets.append(y)
    return torch.cat(preds), torch.cat(targets)


preds, targets = predict_all(vgg, test_loader)
per_class = [(preds[targets == c] == c).float().mean().item() for c in range(10)]
cm = confusion_matrix(to_np(targets), to_np(preds), labels=list(range(10)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
axes[0].bar(CLASSES, [100 * a for a in per_class])
axes[0].set(ylabel="accuracy (%)", title=f"per-class test accuracy (overall {100 * (preds == targets).float().mean():.1f}%)")
axes[0].tick_params(axis="x", rotation=45)
im = axes[1].imshow(cm, cmap="Blues")
axes[1].set(xticks=range(10), yticks=range(10), xticklabels=CLASSES, yticklabels=CLASSES,
            xlabel="predicted", ylabel="true", title="confusion matrix")
axes[1].tick_params(axis="x", rotation=45); axes[1].grid(False)
for i in range(10):
    for j in range(10):
        axes[1].text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.show()

The off-diagonal mass concentrates in a few blocks: cat/dog, automobile/truck, bird/airplane and
deer/horse. These pairs share shape and context (four legs on grass, a wheeled box on a road), and at
$32 \times 32$ pixels the distinguishing detail is often simply not there. When you analyse a model, this
kind of table is worth more than a single accuracy number: it tells you where more data, higher
resolution or a better model would pay off.

## 6. Residual networks

### 6.1 The degradation problem

VGG showed that depth helps, so the obvious next step was to go deeper still: 30, 50, 100 layers. It did
not work. He et al. (2016) documented a puzzling phenomenon they called the **degradation problem**: a
56-layer plain network has *higher training error* than a 20-layer one on CIFAR-10, and a 34-layer plain
network has higher training error than an 18-layer one on ImageNet. This is not overfitting (the training error is what gets worse) and, since the networks used BatchNorm, it is
not simply vanishing gradients either. In principle the deeper network could imitate the shallower one by
setting its extra layers to the identity; in practice gradient descent does not find that solution. The
optimisation landscape of very deep plain networks is just hard.

The experiment below reproduces the effect at small scale. We build "plain" networks in the style of the
ResNet paper's CIFAR experiments: a $3 \times 3$ stem, three stages with $2n$ convolutions each (widths
$w, 2w, 4w$, resolution $32, 16, 8$; the first convolution of stages 2 and 3 has stride 2), global average
pooling and a linear layer, for $6n + 2$ layers in total. We train an 8-layer ($n = 1$) and a 56-layer
($n = 9$) plain network with the *same* optimiser, constant learning rate and data, and plot the training
loss per iteration. Under `FAST_RUN` the budget is a single epoch on 2,000 images with narrow (8-channel)
networks, enough to see the deep network fall behind; the full run uses 20,000 images and 8 epochs.

In [ ]:
def make_plain_cnn(widths=(16, 32, 64), blocks_per_stage=3, num_classes=10):
    """A 'plain' CIFAR network with 6n + 2 layers: stem, 3 stages of 2n conv3x3-BN-ReLU, GAP, linear."""
    layers = [conv_bn_relu(3, widths[0])]
    in_ch = widths[0]
    for i, w in enumerate(widths):
        for b in range(2 * blocks_per_stage):
            stride = 2 if (i > 0 and b == 0) else 1
            layers.append(conv_bn_relu(in_ch, w, stride))
            in_ch = w
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_ch, num_classes)]
    model = nn.Sequential(*layers)
    model.apply(init_weights)
    return model


DEG_WIDTH = 8 if FAST_RUN else 16
DEG_N = 2_000 if FAST_RUN else 20_000
DEG_EPOCHS = 1 if FAST_RUN else 8
deg_widths = (DEG_WIDTH, 2 * DEG_WIDTH, 4 * DEG_WIDTH)
deg_train_loader, deg_test_loader = make_cifar_loaders(DEG_N, N_TEST, batch_size=64)

deg_hist = {}
for name, n in [("plain-8", 1), ("plain-56", 9)]:
    torch.manual_seed(SEED)
    model = make_plain_cnn(deg_widths, blocks_per_stage=n)
    print(f"--- {name}: {count_params(model):,} parameters")
    deg_hist[name] = fit(model, deg_train_loader, deg_test_loader, DEG_EPOCHS, lr=0.05, scheduler_fn=None)

The curves are noisy iteration by iteration, so we smooth them with a moving average, which is (of
course) a 1-D convolution with a box kernel, `np.convolve` from Section 2.1 put to work.

In [ ]:
def smooth(values, window):
    """Moving average with a box kernel; returns fewer points than it receives."""
    values = np.asarray(values, dtype=np.float64)
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="valid")


def plot_iter_losses(hist_dict, window, title):
    plt.figure(figsize=(8, 4))
    for name, h in hist_dict.items():
        plt.plot(smooth(h["iter_loss"], window), label=name)
    plt.xlabel("iteration"); plt.ylabel(f"training loss (moving average, window {window})")
    plt.title(title); plt.legend(); plt.tight_layout(); plt.show()


SMOOTH_WINDOW = 5 if FAST_RUN else 50
plot_iter_losses(deg_hist, SMOOTH_WINDOW, "The degradation problem: deeper plain network, higher training loss")
for name, h in deg_hist.items():
    print(f"{name:10s} final train acc {100 * h['train_acc'][-1]:5.1f}%   test acc {100 * h['test_acc'][-1]:5.1f}%")

The 56-layer plain network trains more slowly and ends with a *higher training loss* than the 8-layer one,
despite having seven times more layers and every opportunity to do at least as well. In the full run the
gap persists for the whole 8 epochs. Something about depth itself makes optimisation harder, and the
fix, once found, was almost embarrassingly simple.

### 6.2 The residual block

Suppose a group of layers is supposed to compute some mapping $H(x)$. He et al. proposed to let the layers
compute the **residual** $F(x) = H(x) - x$ instead and to add the input back:

$$
y = F(x; W) + x .
$$

The addition is the **skip connection** (or shortcut, or identity path). It has no parameters and costs
one element-wise addition. Why does it help so much?

**Identity is easy.** If the best thing a block can do is nothing, a plain block must learn to be the
identity, which for a stack of conv-BN-ReLU layers is a delicate configuration of weights. A residual
block only needs $F(x) = 0$, which is what weight decay pushes it towards anyway. A deeper residual
network can therefore always do at least as well as a shallower one, which removes the degradation problem
by construction.

**Gradients flow through the shortcut.** Consider $L$ residual blocks in a row,
$x_{l+1} = x_l + F_l(x_l)$. Unrolling the recursion gives

$$
x_L = x_l + \sum_{i=l}^{L-1} F_i(x_i),
$$

so the deep feature is the shallow feature *plus* a sum of residuals rather than a product of
transformations. Differentiating the loss with respect to $x_l$ by the chain rule,

$$
\frac{\partial \mathcal{L}}{\partial x_l}
= \frac{\partial \mathcal{L}}{\partial x_L}\,\frac{\partial x_L}{\partial x_l}
= \frac{\partial \mathcal{L}}{\partial x_L}\left(1 + \frac{\partial}{\partial x_l} \sum_{i=l}^{L-1} F_i(x_i)\right).
$$

The term $\partial \mathcal{L} / \partial x_L$ reaches every earlier layer *directly*, multiplied by 1. In a
plain network the same gradient is a product of $L - l$ Jacobians, each of which can shrink or amplify it
(Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization showed
the vanishing/exploding gradient problem in exactly this form). The residual gradient
cannot vanish unless the summed residual derivatives happen to be exactly $-1$, which does not occur in
practice (He et al., 2016, *Identity Mappings in Deep Residual Networks*).

**What goes inside $F$.** The **basic block** used by ResNet-18 and ResNet-34 is two $3 \times 3$
convolutions, each with BatchNorm, with a ReLU between them; the block output is $\mathrm{ReLU}(F(x) + x)$.
The deeper ResNets (50, 101, 152) use a **bottleneck block**: $1 \times 1$ (reduce channels 4x),
$3 \times 3$, $1 \times 1$ (restore), which has the same cost as the basic block but four times as many
output channels.

**When the shapes do not match.** At the start of each stage the resolution halves and the width doubles,
so $x$ and $F(x)$ have different shapes and cannot be added. The paper's option B, used by torchvision,
is a **projection shortcut**: a $1 \times 1$ convolution with stride 2 (Section 3.4) followed by BatchNorm,
which changes the channel count and resolution with the fewest possible parameters. Option A, zero-padding
the extra channels and subsampling, is parameter-free and works almost as well.

> **Key idea.** $y = F(x) + x$ makes "do nothing" the default and gives every layer a direct gradient path
> to the loss. This one change lets networks with hundreds of layers train as easily as networks with ten.

### 6.3 Implementation

`BasicBlock` follows the description above. `ResNetCIFAR` assembles a stem (a single $3 \times 3$
convolution; the ImageNet ResNets use a $7 \times 7$ stride-2 convolution plus max pooling because their
input is $224 \times 224$, which would destroy a $32 \times 32$ image), a list of stages with a
configurable number of blocks and widths, global average pooling and a linear layer. The `block` argument
lets Exercise 3 swap in a block with squeeze-and-excitation. Then we train ResNet-56, the residual twin of
the plain-56 network, with the very same budget, and add it to the plot.

In [ ]:
class BasicBlock(nn.Module):
    """ResNet basic block: conv3x3-BN-ReLU-conv3x3-BN, plus an identity or projection shortcut, then ReLU."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        if stride != 1 or in_ch != out_ch:                      # projection shortcut (option B)
            self.shortcut = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                                          nn.BatchNorm2d(out_ch))
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class ResNetCIFAR(nn.Module):
    """A ResNet for 32x32 inputs: 3x3 stem, len(widths) stages of `blocks[i]` blocks, GAP, linear."""

    def __init__(self, widths=(16, 32, 64), blocks=(3, 3, 3), num_classes=10, block=BasicBlock):
        super().__init__()
        self.stem = conv_bn_relu(3, widths[0])
        stages, in_ch = [], widths[0]
        for i, (w, n) in enumerate(zip(widths, blocks)):
            stride = 1 if i == 0 else 2
            layers = [block(in_ch, w, stride)] + [block(w, w, 1) for _ in range(n - 1)]
            stages.append(nn.Sequential(*layers))
            in_ch = w
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_ch, num_classes))
        self.apply(init_weights)

    def forward(self, x):
        return self.head(self.stages(self.stem(x)))


torch.manual_seed(SEED)
resnet56 = ResNetCIFAR(deg_widths, blocks=(9, 9, 9))
x = torch.zeros(1, 3, 32, 32)
with torch.no_grad():
    h = resnet56.stem(x)
    print("stem   ->", tuple(h.shape))
    for i, stage in enumerate(resnet56.stages):
        h = stage(h)
        print(f"stage {i + 1} -> {tuple(h.shape)}   ({len(stage)} blocks)")
    print("head   ->", tuple(resnet56.head(h).shape))
print(f"\n--- ResNet-56: {count_params(resnet56):,} parameters")
deg_hist["ResNet-56"] = fit(resnet56, deg_train_loader, deg_test_loader, DEG_EPOCHS, lr=0.05, scheduler_fn=None)

plot_iter_losses(deg_hist, SMOOTH_WINDOW, "Skip connections fix the degradation problem")
for name, h in deg_hist.items():
    print(f"{name:10s} final train acc {100 * h['train_acc'][-1]:5.1f}%   test acc {100 * h['test_acc'][-1]:5.1f}%")

ResNet-56 has essentially the same layers and parameter count as plain-56 (the two projection shortcuts
add fewer than 3,000 weights), yet its training loss drops at least as fast as that of the shallow plain-8
network and, given enough epochs, well below it. Depth is useful after all; it was the *optimisation* of
plain deep networks that was broken, and skip connections fix it.

> **Going deeper.** Veit et al. (2016) offer a complementary view: unrolling $x_L = x_0 + \sum F_i$ shows
> that a ResNet behaves like an ensemble of exponentially many shallow paths, and most of the gradient
> flows through paths of only 10-30 layers even in a 110-layer network. Deleting a single block from a
> trained ResNet barely hurts it; deleting one from a plain network is fatal.

### 6.4 A ResNet-18-like network on CIFAR-10

We now build the model we will keep for the rest of the notebook: four stages of two basic blocks, the
layout of ResNet-18, with widths reduced from the ImageNet (64, 128, 256, 512) to (32, 64, 128, 256) in
the full run and (16, 32, 64, 128) under `FAST_RUN`, so that it trains in the same time as the VGG-style
network. We train it with the same loaders, epochs and optimiser settings and compare.

In [ ]:
RES_WIDTH = 16 if FAST_RUN else 32
res_widths = (RES_WIDTH, 2 * RES_WIDTH, 4 * RES_WIDTH, 8 * RES_WIDTH)
torch.manual_seed(SEED)
resnet = ResNetCIFAR(res_widths, blocks=(2, 2, 2, 2))
print(f"ResNet-18-like, widths {res_widths}: {count_params(resnet):,} parameters, "
      f"{EPOCHS_CIFAR} epoch(s) on {N_TRAIN:,} images")
hist_resnet = fit(resnet, train_loader, test_loader, epochs=EPOCHS_CIFAR, lr=0.05)
plot_history({"VGG-style": hist_vgg, "ResNet-18-like": hist_resnet}, title="VGG-style vs ResNet on CIFAR-10")

print(f"\n{'model':16s} {'params':>10s} {'s/epoch':>8s} {'best test acc':>14s}")
for name, model, h in [("VGG-style", vgg, hist_vgg), ("ResNet-18-like", resnet, hist_resnet)]:
    print(f"{name:16s} {count_params(model):>10,d} {np.mean(h['epoch_time']):8.1f} {100 * max(h['test_acc']):13.1f}%")

In the full run the ResNet reaches roughly 88-91 % against 85-88 % for the VGG-style network, with a
similar number of parameters and a similar time per epoch. Twelve epochs are far from convergence for
either model (the standard 200-epoch schedule takes ResNet-18 to about 95 %), and the exercises show two
ways to squeeze more out of the same budget (squeeze-and-excitation, one-cycle with label smoothing).
Under `FAST_RUN`, both models are barely past random initialisation and their ranking is not meaningful.

### 6.5 Where does BatchNorm go? Pre-activation blocks

The original block computes $\mathrm{ReLU}(\mathrm{BN}(\mathrm{conv}(\cdots)) + x)$: the shortcut is
added *before* the final ReLU, and BatchNorm sits after each convolution. He et al. (2016b) studied
several alternatives and found that moving both BatchNorm and ReLU *in front of* each convolution,
the **pre-activation** block

$$
y = x + \mathrm{conv}\big(\mathrm{ReLU}(\mathrm{BN}(\mathrm{conv}(\mathrm{ReLU}(\mathrm{BN}(x)))))\big),
$$

leaves the identity path completely clean (no ReLU on it) and trains better for very deep networks
(ResNet-1001). For networks of ResNet-18 to ResNet-50 depth the difference is small, and torchvision keeps
the original layout, but pre-activation is the design most later architectures (Wide ResNets, many
diffusion U-Nets) build on. Here is the block; we only check that it plugs into `ResNetCIFAR`.

In [ ]:
class PreActBlock(nn.Module):
    """Pre-activation basic block (He et al., 2016b): BN-ReLU-conv-BN-ReLU-conv, clean identity path."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.shortcut = None
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)

    def forward(self, x):
        pre = F.relu(self.bn1(x))
        shortcut = x if self.shortcut is None else self.shortcut(pre)   # projection uses the pre-activated input
        out = self.conv1(pre)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


torch.manual_seed(SEED)
preact = ResNetCIFAR(res_widths, blocks=(2, 2, 2, 2), block=PreActBlock)
with torch.no_grad():
    print("pre-activation ResNet:", tuple(preact(torch.zeros(2, 3, 32, 32)).shape),
          f"| {count_params(preact):,} parameters")

(A complete pre-activation network would add one final BatchNorm and ReLU before the global pooling,
because the last block's output is a raw sum; our shape check leaves this aside.)

> **Try it.** Train `preact` with `fit(...)` on the same loaders and compare with `hist_resnet`. At this
> depth expect the two to be within noise of each other.

## 7. A tour of architectures

Between 2012 and 2022 the ImageNet benchmark (1.28 million training images, 1,000 classes) drove a
sequence of architectures, each of which introduced one or two ideas that everything after it kept. The
table lists representative members (year of the first preprint) with approximate single-crop top-1
accuracy on ImageNet, parameter counts and the cost of one forward pass in giga-FLOPs at
$224 \times 224$, all as reported for torchvision's original weights and rounded.

| year | architecture | depth | params | GFLOPs | top-1 | what it added |
|---|---|---|---|---|---|---|
| 2012 | AlexNet | 8 | 61 M | 0.7 | 57 % | ReLU, dropout, GPU training, augmentation; the deep learning breakthrough |
| 2014 | VGG-16 | 16 | 138 M | 15.5 | 72 % | only 3x3 convolutions, stages that halve resolution and double width |
| 2014 | GoogLeNet (Inception v1) | 22 | 6.6 M | 1.5 | 70 % | multi-branch Inception module, 1x1 bottlenecks, global average pooling |
| 2015 | ResNet-50 | 50 | 25.6 M | 4.1 | 76 % | residual connections, bottleneck blocks, BatchNorm everywhere |
| 2016 | ResNeXt-50 (32x4d) | 50 | 25 M | 4.2 | 78 % | grouped convolutions ("cardinality") inside the bottleneck |
| 2016 | DenseNet-121 | 121 | 8.0 M | 2.8 | 74 % | every layer receives all earlier feature maps (concatenation) |
| 2018 | MobileNetV2 | 53 | 3.5 M | 0.3 | 72 % | depthwise-separable convolutions, inverted residuals, for phones |
| 2019 | EfficientNet-B0 | 82 | 5.3 M | 0.4 | 78 % | compound scaling of depth/width/resolution, squeeze-and-excitation, NAS |
| 2022 | ConvNeXt-T | 18 blocks | 28.6 M | 4.5 | 83 % | a ResNet modernised with Transformer-era design choices |

**AlexNet** (Krizhevsky, Sutskever and Hinton, 2012) is LeNet scaled up: five convolutions and three
linear layers, trained on two GPUs for a week. Its 10-point margin over the best hand-engineered
computer-vision pipeline started the modern era. **VGG** made the design regular; its cost shows why
nobody uses it in production. **GoogLeNet** asked how to be accurate *and* cheap: its Inception module runs
$1 \times 1$, $3 \times 3$ and $5 \times 5$ convolutions in parallel and concatenates them, using
$1 \times 1$ bottlenecks to keep the cost down, and it replaced the large linear layers by global average
pooling; about nine times fewer parameters than AlexNet with better accuracy.

**ResNet** (Section 6) is the watershed: after it, depth stopped being a problem, and the residual block
became the default building unit for vision, language (Transformers are residual networks, Notebook 09 ·
Attention and Transformers) and generative models. **ResNeXt** split the bottleneck's $3 \times 3$ convolution into 32 parallel groups
(`groups=32`), a middle ground between a full convolution and a depthwise one. **DenseNet** took the
skip-connection idea to its limit by concatenating rather than adding, which encourages feature reuse and
gives good accuracy per parameter at the price of memory.

The **MobileNet** family (Section 3.5) and **EfficientNet** optimise for a different objective, accuracy
per FLOP on mobile hardware: depthwise-separable convolutions, "inverted" residual blocks that expand the
channel count inside the block, squeeze-and-excitation channel attention (Exercise 3) and, in
EfficientNet, a principled rule for scaling depth, width and input resolution together. Finally,
**ConvNeXt** (Liu et al., 2022) asked whether the 2020-era success of Vision Transformers (Notebook 07 ·
Transfer Learning and Modern Vision) was due to attention or to the surrounding design choices; by transplanting them into a ResNet (larger
$7 \times 7$ depthwise kernels, fewer activations and normalisations, LayerNorm and GELU, a patchify
stem) they matched Swin Transformers with a pure CNN.

In [ ]:
tour = [("AlexNet", 0.71, 56.5), ("VGG-16", 15.5, 71.6), ("GoogLeNet", 1.5, 69.8), ("ResNet-50", 4.1, 76.1),
        ("ResNeXt-50", 4.2, 77.6), ("DenseNet-121", 2.8, 74.4), ("MobileNetV2", 0.3, 71.9),
        ("EfficientNet-B0", 0.4, 77.7), ("ConvNeXt-T", 4.5, 82.5)]
plt.figure(figsize=(8, 4.5))
for name, gflops, acc in tour:
    plt.scatter(gflops, acc, s=40)
    plt.annotate(name, (gflops, acc), textcoords="offset points", xytext=(5, 4), fontsize=8)
plt.xscale("log"); plt.xlabel("forward-pass GFLOPs at 224x224 (log scale)"); plt.ylabel("ImageNet top-1 (%)")
plt.title("Accuracy versus compute across a decade of CNNs"); plt.tight_layout(); plt.show()

Read the plot from left to right for the same accuracy: MobileNetV2 matches VGG-16 with 50x fewer FLOPs,
and EfficientNet-B0 matches ResNet-50 with 10x fewer. Read it bottom to top for the same compute: at
about 4 GFLOPs, ResNet-50, ResNeXt-50 and ConvNeXt-T span six points of accuracy. Architecture research is
the art of moving up and to the left on this plot; since 2020, larger datasets and pretraining (Notebook 07)
have mattered at least as much as architecture.

## 8. Understanding CNNs

A trained network is not a black box if you know where to look. Three classic tools, all cheap to
implement, are: drawing the first-layer filters, showing feature maps at several depths, and measuring
how the prediction changes when parts of the input are hidden.

### 8.1 First-layer filters

The stem of our ResNet has $3 \times 3$ kernels with three input channels, so each kernel is a tiny RGB
image. We rescale each to $[0, 1]$ for display.

In [ ]:
w_stem = to_np(resnet.stem[0].weight)                           # (RES_WIDTH, 3, 3, 3)
n_show = min(16, w_stem.shape[0])
fig, axes = plt.subplots(2, n_show // 2, figsize=(12, 3.4))
for i, ax in enumerate(axes.flat):
    k = w_stem[i].transpose(1, 2, 0)                             # (3, 3, 3) -> (H, W, C)
    k = (k - k.min()) / (k.max() - k.min() + 1e-8)
    ax.imshow(k, interpolation="nearest"); ax.set_title(f"filter {i}", fontsize=8); ax.axis("off")
plt.suptitle("ResNet stem: first-layer 3x3 RGB kernels"); plt.tight_layout(); plt.show()

Even at $3 \times 3$ and after a short training run, you can recognise the two families that every
vision network learns first: **oriented edge detectors** (a bright and a dark side, all three channels
alike) and **colour-opponent detectors** (for example green versus magenta), the same kind of pattern that
Section 2.5 designed by hand. Kernels that still look like random noise have not been trained long enough
or are not needed; in the full run their share is much smaller than under `FAST_RUN`.

### 8.2 Feature maps at several depths

Forward hooks (Notebook 05 · The PyTorch Workflow and Best Practices) let us capture the output of any
module without changing the model. We hook
the stem and each of the four stages, run one test image through the network and show eight channels of
each feature map. Resolution halves and the maps become more abstract with depth.

In [ ]:
def collect_feature_maps(model, x):
    """Return {name: output tensor} for the stem and each stage of a ResNetCIFAR, for one input batch."""
    feats, handles = {}, []

    def make_hook(name):
        def hook(module, inputs, output):
            feats[name] = output.detach()
        return hook

    handles.append(model.stem.register_forward_hook(make_hook("stem")))
    for i, stage in enumerate(model.stages):
        handles.append(stage.register_forward_hook(make_hook(f"stage {i + 1}")))
    model.eval()
    with torch.no_grad():
        model(x.to(device))
    for h in handles:
        h.remove()
    return feats


idx_img = 3
x_img, y_img = cifar_test_x[idx_img], cifar_test_y[idx_img].item()
feats = collect_feature_maps(resnet, eval_tf(x_img)[None])

fig, axes = plt.subplots(5, 9, figsize=(14, 8))
axes[0, 0].imshow(to_np(x_img.permute(1, 2, 0))); axes[0, 0].set_title(f"input: {CLASSES[y_img]}", fontsize=9)
for r, (name, fm) in enumerate(feats.items()):
    fm = fm[0]                                                   # (C, H, W)
    axes[r, 0].set_ylabel(f"{name}\n{tuple(fm.shape)}", fontsize=8)
    if r > 0:
        axes[r, 0].imshow(to_np(fm.mean(dim=0)), cmap="viridis"); axes[r, 0].set_title("channel mean", fontsize=8)
    for c in range(8):
        axes[r, c + 1].imshow(to_np(fm[c]), cmap="viridis")
        axes[r, c + 1].set_title(f"ch {c}", fontsize=8)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.show()

The stem and stage-1 maps are recognisably filtered versions of the image (edges, colour blobs, the
object's silhouette). Stage 2 keeps the layout but responds to parts and textures; by stages 3 and 4 the
maps are $8 \times 8$ and $4 \times 4$ grids of abstract activations that are impossible to interpret by
eye but that the linear classifier separates almost linearly. This is the feature hierarchy that Zeiler
and Fergus (2014) documented for AlexNet, and it is the reason transfer learning works: the early layers
are generic and can be reused on any natural-image task (Notebook 07 · Transfer Learning and Modern
Vision).

### 8.3 Occlusion sensitivity

Which part of the image does the prediction depend on? The simplest answer is to hide parts of the image
and watch the prediction change (Zeiler and Fergus, 2014). We slide a grey square (the value 0 in
normalised space, i.e. the dataset mean colour) across the image with a small stride, run the network on
every occluded copy in one batch, and record the probability of the true class. Regions where hiding the
patch makes the probability collapse are the evidence the network relies on. We pick the first test image
the network classifies correctly with reasonable confidence; the more accurate the model, the sharper the map.

In [ ]:
@torch.no_grad()
def occlusion_map(model, x_norm, target, patch=8, stride=2, fill=0.0):
    """Probability of `target` when a patch x patch square is occluded at each position (grid of positions)."""
    model.eval()
    _, H, W = x_norm.shape
    rows = list(range(0, H - patch + 1, stride))
    cols = list(range(0, W - patch + 1, stride))
    batch = x_norm[None].repeat(len(rows) * len(cols), 1, 1, 1).clone()
    for k, (i, j) in enumerate((i, j) for i in rows for j in cols):
        batch[k, :, i:i + patch, j:j + patch] = fill
    with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
        logits = model(batch.to(device))
    probs = F.softmax(logits.float(), dim=1)[:, target].cpu()
    return probs.view(len(rows), len(cols))


# choose the first confidently correct image among the first 256 test images
resnet.eval()
with torch.no_grad():
    xs = torch.stack([eval_tf(cifar_test_x[i]) for i in range(256)])
    probs = F.softmax(resnet(xs.to(device)).float(), dim=1).cpu()
conf, pred = probs.max(dim=1)
candidates = torch.nonzero((pred == cifar_test_y[:256]) & (conf > 0.6)).flatten()
chosen = candidates[0].item() if len(candidates) > 0 else 0
x_img, y_img = cifar_test_x[chosen], cifar_test_y[chosen].item()
x_norm = eval_tf(x_img)
with torch.no_grad():
    p_full = F.softmax(resnet(x_norm[None].to(device)).float(), dim=1)[0, y_img].item()
occ = occlusion_map(resnet, x_norm, y_img, patch=8, stride=2)
occ_up = F.interpolate(occ[None, None], size=(32, 32), mode="bilinear", align_corners=False)[0, 0]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].imshow(to_np(x_img.permute(1, 2, 0))); axes[0].set_title(f"test image {chosen}: {CLASSES[y_img]} (p = {p_full:.2f})")
im = axes[1].imshow(to_np(occ), cmap="RdYlGn", vmin=0, vmax=1)
axes[1].set_title("P(true class) with patch occluded"); plt.colorbar(im, ax=axes[1], fraction=0.046)
axes[2].imshow(to_np(x_img.permute(1, 2, 0)))
axes[2].imshow(to_np(p_full - occ_up), cmap="jet", alpha=0.5, vmin=0, vmax=max(p_full, 1e-3))
axes[2].set_title("drop in probability (overlay)")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

Red cells in the middle panel mark positions where covering an $8 \times 8$ patch destroys the prediction;
green cells are positions the network does not care about. For a well-trained model the red region sits
on the object (the head of an animal, the body of a vehicle) and the background is green, which is the
behaviour we want. When the map highlights the background instead (snow for "deer", water for "ship"),
the model has learned a *shortcut*, and this is precisely the kind of failure that explanation methods
exist to catch. Occlusion is slow (one forward pass per position) and coarse; Notebook 07 implements
Grad-CAM, which obtains a similar map from a single backward pass, and Notebook 16 · Interpretability,
Robustness and Uncertainty covers attribution methods systematically.

> **Try it.** Change `patch` to 4 and `stride` to 1 for a finer map (841 positions instead of 169), or set
> `fill` to a large value to occlude with white instead of grey and see whether the map changes.

## Exercises

### Exercise 1 (warm-up): dilated convolution from scratch

Extend `corr2d` with a `dilation` argument, so that the kernel taps are spaced `dilation` pixels apart
(a $3 \times 3$ kernel with dilation 2 covers a $5 \times 5$ window). Verify your implementation against
`F.conv2d(..., dilation=d)` for $d \in \{1, 2, 3\}$ with padding and stride, and check that the output
size matches `conv_out_size`.

**Solution to Exercise 1**

The only change is in how the patch is read: with dilation $d$ the window spans $d(k-1)+1$ pixels, and
we take every $d$-th pixel of it with a strided slice.

In [ ]:
def corr2d_dilated(x, w, stride=1, padding=0, dilation=1):
    """2-D cross-correlation with stride, zero padding and dilation (single channel)."""
    if padding > 0:
        x = np.pad(x, padding)
    H, W = x.shape
    kh, kw = w.shape
    eff_h, eff_w = dilation * (kh - 1) + 1, dilation * (kw - 1) + 1     # effective kernel size
    H_out, W_out = (H - eff_h) // stride + 1, (W - eff_w) // stride + 1
    y = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            r, c = i * stride, j * stride
            patch = x[r:r + eff_h:dilation, c:c + eff_w:dilation]       # every d-th pixel of the window
            y[i, j] = np.sum(patch * w)
    return y


x_np = rng.standard_normal((13, 13)).astype(np.float32)
w_np = rng.standard_normal((3, 3)).astype(np.float32)
for d, s, p in [(1, 1, 1), (2, 1, 2), (3, 2, 0), (2, 2, 1)]:
    ours = corr2d_dilated(x_np, w_np, stride=s, padding=p, dilation=d)
    ref = F.conv2d(torch.tensor(x_np)[None, None], torch.tensor(w_np)[None, None],
                   stride=s, padding=p, dilation=d)[0, 0]
    expected = conv_out_size(13, 3, stride=s, padding=p, dilation=d)
    print(f"dilation={d} stride={s} padding={p}: shape {ours.shape} (formula {expected}), "
          f"max |ours - F.conv2d| = {np.abs(ours - to_np(ref)).max():.1e}")

All four configurations match PyTorch to `float32` precision, and the output size agrees with the formula
of Section 2.3, which already contained the $d(k-1)+1$ effective size.

### Exercise 2: receptive fields of the networks we trained

Write a function that extracts the `(kernel_size, stride)` sequence from an `nn.Sequential` of
convolution and pooling layers and use it with `receptive_field` to compute the receptive field of
LeNet-5's feature extractor and of the plain-8 network. Then compute the receptive field along the main
path of the ResNet-18-like model after each stage (ignore the shortcuts: they never enlarge the receptive
field). After which stage does one output pixel see the whole $32 \times 32$ image?

**Solution to Exercise 2**

For sequential models we walk over `model.modules()` and keep convolutions and pooling layers. For the
ResNet we take `conv1` and `conv2` of every block in order; the projection shortcut is a $1 \times 1$
convolution with the same stride, whose receptive field is contained in the main path's, so it can be
ignored for the size computation.

In [ ]:
def layers_from_sequential(model):
    """(kernel_size, stride) of every Conv2d / pooling layer in a sequential model, in order."""
    layers = []
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            k = m.dilation[0] * (m.kernel_size[0] - 1) + 1
            layers.append((k, m.stride[0]))
        elif isinstance(m, (nn.MaxPool2d, nn.AvgPool2d)):
            k = m.kernel_size if isinstance(m.kernel_size, int) else m.kernel_size[0]
            s = m.stride if isinstance(m.stride, int) else m.stride[0]
            layers.append((k, s))
    return layers


def resnet_main_path(model):
    """(kernel_size, stride) along the main path of a ResNetCIFAR, plus the index where each stage ends."""
    layers, stage_ends = [(3, 1)], []
    for stage in model.stages:
        for block in stage:
            layers.append((block.conv1.kernel_size[0], block.conv1.stride[0]))
            layers.append((block.conv2.kernel_size[0], block.conv2.stride[0]))
        stage_ends.append(len(layers) - 1)
    return layers, stage_ends


for name, seq in [("LeNet-5 features", lenet.features), ("plain-8", make_plain_cnn(deg_widths, 1))]:
    rf = receptive_field(layers_from_sequential(seq))
    print(f"{name:18s} layers {layers_from_sequential(seq)}\n{'':18s} RF after each layer {[r for r, _ in rf]}")

layers, stage_ends = resnet_main_path(resnet)
rf = receptive_field(layers)
print("\nResNet-18-like main path:")
print(f"  after stem   : RF {rf[0][0]:3d}, jump {rf[0][1]}")
for i, end in enumerate(stage_ends):
    print(f"  after stage {i + 1}: RF {rf[end][0]:3d}, jump {rf[end][1]}")

LeNet's last feature map has a receptive field of 16 pixels (of 28), which is why it needs the linear
layers to combine evidence from the whole digit. The plain-8 network reaches 25. In the ResNet-18-like
model the receptive field is 11 after stage 1, 25 after stage 2 and 53 after stage 3, so from stage 3 on
every output pixel can in principle see the entire $32 \times 32$ image, and after stage 4 (109) the
theoretical receptive field is three times the image; the *effective* receptive field (Section 2.6) is
of course much smaller.

### Exercise 3: squeeze-and-excitation blocks

Hu et al. (2018) added a cheap channel-attention mechanism to residual blocks. A **squeeze-and-excitation
(SE)** block computes, from a feature map $u \in \mathbb{R}^{C \times H \times W}$,

$$
z = \frac{1}{HW}\sum_{i,j} u_{:, i, j} \in \mathbb{R}^{C} \quad\text{(squeeze: global average pooling)}, \qquad
s = \sigma\!\left(W_2\, \mathrm{ReLU}(W_1 z)\right) \in (0, 1)^C \quad\text{(excitation)},
$$

with $W_1 \in \mathbb{R}^{C/r \times C}$, $W_2 \in \mathbb{R}^{C \times C/r}$ and reduction ratio $r = 16$, and
rescales every channel: $\tilde u_c = s_c\, u_c$. Implement `SEBlock`, a `SEBasicBlock` that applies it to
the residual branch before the addition, build an SE-ResNet with `ResNetCIFAR(..., block=SEBasicBlock)`,
and compare it with the plain ResNet of the same size on a short training run. How many parameters does
SE add?

**Solution to Exercise 3**

Because `BasicBlock` already builds the convolutions and the shortcut, `SEBasicBlock` only needs to add
the SE module and override `forward`. We train both variants on a small ResNet (widths 16/32/64, two
blocks per stage) with a reduced budget so that the exercise stays cheap.

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-excitation: global average pool -> FC -> ReLU -> FC -> sigmoid -> rescale channels."""

    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)

    def forward(self, x):
        z = x.mean(dim=(2, 3))                                   # squeeze: (N, C)
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(z))))        # excitation: (N, C) in (0, 1)
        return x * s[:, :, None, None]                           # rescale every channel


class SEBasicBlock(BasicBlock):
    """Basic block with squeeze-and-excitation on the residual branch."""

    def __init__(self, in_ch, out_ch, stride=1, reduction=16):
        super().__init__(in_ch, out_ch, stride)
        self.se = SEBlock(out_ch, reduction)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.se(self.bn2(self.conv2(out)))
        return F.relu(out + self.shortcut(x))


EX_N = 2_000 if FAST_RUN else 20_000
EX_EPOCHS = 1 if FAST_RUN else 6
ex_train_loader, ex_test_loader = make_cifar_loaders(EX_N, N_TEST, batch_size=128)
small_widths, small_blocks = (16, 32, 64), (2, 2, 2)

hist_se = {}
for name, block in [("ResNet (small)", BasicBlock), ("SE-ResNet (small)", SEBasicBlock)]:
    torch.manual_seed(SEED)
    model = ResNetCIFAR(small_widths, small_blocks, block=block)
    print(f"--- {name}: {count_params(model):,} parameters")
    hist_se[name] = fit(model, ex_train_loader, ex_test_loader, EX_EPOCHS, lr=0.05)
plot_history(hist_se, title="Squeeze-and-excitation on a small CIFAR ResNet")

SE adds two small linear layers per block, $2 C^2 / r$ weights, which for this network is under 2 % of
the parameters; the extra compute is negligible because it acts on pooled vectors. In the full run the SE
variant is typically 0.5-1.5 points better after 6 epochs; a difference of this size needs several seeds
to be conclusive, and under `FAST_RUN` the comparison is meaningless. The mechanism, letting the network
re-weight channels based on global context, is the ancestor of the attention mechanisms of Notebook 09 ·
Attention and Transformers.

### Exercise 4: Cutout augmentation

Cutout (DeVries and Taylor, 2017) zeroes out one random square of the (normalised) training image, which
forces the network to use evidence from the whole object instead of its most distinctive part. Implement
it as a callable that can be appended to the `transforms.v2` pipeline, show a few augmented samples, and
train the small ResNet with Cutout using the same budget as Exercise 3. Which patch size is reasonable for
$32 \times 32$ images?

**Solution to Exercise 4**

`v2.Compose` accepts any callable, so a plain class with `__call__` works. The centre of the square is
sampled uniformly over the image and the square is clipped at the border, exactly as in the paper (which
uses a side of 16 pixels for CIFAR-10, i.e. up to a quarter of the image; we use 12 for the short run).
Because it comes after `Normalize`, zero corresponds to the mean colour.

In [ ]:
class Cutout:
    """Zero out one random square of side `size` (clipped at the border) in a (C, H, W) tensor."""

    def __init__(self, size=16):
        self.size = size

    def __call__(self, img):
        _, H, W = img.shape
        cy, cx = torch.randint(H, (1,)).item(), torch.randint(W, (1,)).item()
        y0, y1 = max(cy - self.size // 2, 0), min(cy + self.size // 2, H)
        x0, x1 = max(cx - self.size // 2, 0), min(cx + self.size // 2, W)
        img = img.clone()
        img[:, y0:y1, x0:x1] = 0.0
        return img


cutout_tf = v2.Compose([v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip(),
                        v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
                        Cutout(12)])

torch.manual_seed(SEED)
fig, axes = plt.subplots(1, 6, figsize=(12, 2.3))
for ax in axes:
    ax.imshow(denorm(cutout_tf(cifar_train_x[4]))); ax.axis("off")
plt.suptitle("Cutout(12) after crop and flip"); plt.tight_layout(); plt.show()

cut_train_loader, _ = make_cifar_loaders(EX_N, N_TEST, batch_size=128, transform=cutout_tf)
torch.manual_seed(SEED)
model = ResNetCIFAR(small_widths, small_blocks)
hist_se["ResNet (small) + Cutout"] = fit(model, cut_train_loader, ex_test_loader, EX_EPOCHS, lr=0.05)
plot_history({k: hist_se[k] for k in ["ResNet (small)", "ResNet (small) + Cutout"]}, title="Cutout")

Cutout is a regulariser: it raises the training loss and, given enough epochs, lowers the test error
(the paper reports about 1 point on CIFAR-10 for a ResNet-18 trained for 200 epochs). In a short run of
6 epochs the network is still underfitting, so the extra regularisation does not help yet and can even
hurt a little, which is a useful lesson in itself: augmentation strength should match the length of
training. Under `FAST_RUN` a single epoch shows nothing at all. A side of 8-16 pixels is reasonable for
$32 \times 32$ images; much larger squares remove the object entirely too often.

### Exercise 5 (challenge): one-cycle schedule and label smoothing

Two training-recipe improvements popular since 2018 are the **one-cycle learning-rate schedule**
(Smith, 2018): warm up from a small learning rate to a large peak over the first 30 % of the iterations,
then anneal to almost zero, and **label smoothing** (Szegedy et al., 2016): train against the softened
target $q_k = (1 - \epsilon)\,\mathbb{1}[k = y] + \epsilon / K$ instead of a one-hot vector, which stops the
logits from growing without bound and improves calibration.

1. Implement label-smoothing cross-entropy from the definition
   $\ell = -\sum_k q_k \log p_k = (1-\epsilon)(-\log p_y) + \frac{\epsilon}{K}\sum_k (-\log p_k)$
   and verify it against `F.cross_entropy(..., label_smoothing=eps)`.
2. Implement the one-cycle schedule as a `LambdaLR` multiplier (cosine warm-up from `max_lr / 25`, cosine
   annealing to `max_lr / 25 / 1e4`, warm-up over 30 % of the steps) and compare it with
   `torch.optim.lr_scheduler.OneCycleLR`.
3. Train the ResNet-18-like model with both, with the same loaders and epochs as Section 6.4, and
   compare with the baseline `hist_resnet`.

**Solution to Exercise 5**

Part 1 is a direct translation of the formula using `log_softmax` (computed in `float32` so that it is
safe under mixed precision). Part 2 mirrors `OneCycleLR`'s two phases: the phase boundary is at step
$0.3\,T - 1$ and both phases use cosine annealing, so a `LambdaLR` with the right multiplier reproduces
PyTorch's schedule to floating-point precision. Part 3 reuses `fit`, which accepts a custom criterion
and a scheduler factory.

In [ ]:
def smoothed_cross_entropy(logits, targets, eps=0.1):
    """Cross-entropy against (1 - eps) * one_hot + eps / K, averaged over the batch."""
    log_p = F.log_softmax(logits.float(), dim=1)
    nll = -log_p.gather(1, targets[:, None]).squeeze(1)          # -log p_y
    uniform = -log_p.mean(dim=1)                                 # (1/K) sum_k -log p_k
    return ((1 - eps) * nll + eps * uniform).mean()


logits = torch.randn(16, 10)
targets = torch.randint(10, (16,))
ours = smoothed_cross_entropy(logits, targets, eps=0.1)
ref = F.cross_entropy(logits, targets, label_smoothing=0.1)
print(f"label smoothing: ours {ours.item():.6f} | F.cross_entropy {ref.item():.6f} | diff {abs(ours.item() - ref.item()):.1e}")


def one_cycle_lambda(total_steps, pct_start=0.3, div_factor=25.0, final_div_factor=1e4):
    """Multiplier f(step) such that lr = max_lr * f(step) follows torch's OneCycleLR (cosine, two phases)."""
    lo, final = 1.0 / div_factor, 1.0 / (div_factor * final_div_factor)
    end_warm = pct_start * total_steps - 1

    def cos_anneal(start, end, frac):
        return end + (start - end) / 2 * (1 + math.cos(math.pi * frac))

    def f(step):
        if step <= end_warm:
            return cos_anneal(lo, 1.0, step / end_warm)
        return cos_anneal(1.0, final, (step - end_warm) / (total_steps - 1 - end_warm))
    return f


T, MAX_LR = 1000, 0.1
lr_curves = {}
for kind in ["ours", "torch"]:
    p = nn.Parameter(torch.zeros(1))
    opt = torch.optim.SGD([p], lr=MAX_LR, momentum=0.9)
    if kind == "ours":
        sched = torch.optim.lr_scheduler.LambdaLR(opt, one_cycle_lambda(T))
    else:
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=MAX_LR, total_steps=T)
    lrs = []
    for _ in range(T):
        lrs.append(opt.param_groups[0]["lr"])
        opt.step(); sched.step()
    lr_curves[kind] = np.array(lrs)
print(f"one-cycle: max |ours - OneCycleLR| = {np.abs(lr_curves['ours'] - lr_curves['torch']).max():.1e}")
plt.figure(figsize=(7, 3.5))
plt.plot(lr_curves["torch"], label="torch OneCycleLR"); plt.plot(lr_curves["ours"], "--", label="our LambdaLR")
plt.xlabel("iteration"); plt.ylabel("learning rate"); plt.title("One-cycle schedule"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
torch.manual_seed(SEED)
resnet_oc = ResNetCIFAR(res_widths, blocks=(2, 2, 2, 2))
hist_oc = fit(resnet_oc, train_loader, test_loader, epochs=EPOCHS_CIFAR, lr=0.1,
              criterion=lambda logits, y: smoothed_cross_entropy(logits, y, eps=0.1),
              scheduler_fn=lambda opt, total: torch.optim.lr_scheduler.LambdaLR(opt, one_cycle_lambda(total)))
plot_history({"ResNet baseline (cosine)": hist_resnet, "ResNet one-cycle + label smoothing": hist_oc},
             title="Training recipe matters")
print(f"best test accuracy: baseline {100 * max(hist_resnet['test_acc']):.1f}%  |  "
      f"one-cycle + smoothing {100 * max(hist_oc['test_acc']):.1f}%")

Note that the training *loss* of the label-smoothed run is not comparable with the baseline's: the
smoothed objective has a non-zero minimum ($\epsilon$ mass on the wrong classes cannot be driven to zero),
so only the test accuracy should be compared. In the full 12-epoch run the one-cycle recipe with a peak
learning rate of 0.1 typically gains 1-2 points over the cosine baseline, mainly because the warm-up lets
us use a higher peak learning rate safely and the long anneal at the end converges more precisely. On a
200-epoch budget the two recipes converge to similar numbers; the one-cycle schedule is above all a way
to get good results *quickly*.

## Summary

- Images are tensors of shape $(N, C, H, W)$; convolutional layers exploit locality, weight sharing and
  translation equivariance to process them with a parameter count that does not depend on the image size.
- Deep learning's "convolution" is cross-correlation, $Y[i,j] = \sum_{a,b} X[i+a, j+b]\,K[a,b]$, summed
  over input channels; the weight tensor has shape $(C_{\text{out}}, C_{\text{in}}, k, k)$ and
  $C_{\text{out}} C_{\text{in}} k^2 + C_{\text{out}}$ parameters.
- The output size is $\lfloor (n + 2p - d(k-1) - 1)/s \rfloor + 1$; the receptive field grows by
  $(k-1)$ times the accumulated stride per layer, which is why networks alternate convolutions and
  downsampling.
- Convolution is a matrix multiplication after im2col (`F.unfold`); the cost is
  $C_{\text{out}} C_{\text{in}} k^2 H_{\text{out}} W_{\text{out}}$ multiply-accumulates. Depthwise-separable
  convolutions cut this by about $k^2$, $1 \times 1$ convolutions mix channels only, transposed convolutions
  are the adjoint of convolutions and upsample.
- Pooling and strided convolutions downsample; global average pooling replaces large linear classifiers
  and makes networks input-size agnostic.
- LeNet-5 (1998) already had the modern structure: conv-nonlinearity-pool blocks followed by a classifier;
  it reaches about 99 % on MNIST with 62,000 parameters. Its first layer learns oriented edge detectors.
- VGG-style networks use only $3 \times 3$ convolutions in stages that halve resolution and double width;
  with BatchNorm and crop-and-flip augmentation such a network reaches 85 %+ on CIFAR-10 in a few minutes.
- Plain networks beyond about 20 layers suffer from the degradation problem: their *training* loss gets
  worse with depth. Residual blocks $y = F(x) + x$ fix this by making the identity easy and giving every
  layer a direct gradient path; ResNets train at any depth and beat VGG-style networks at equal cost.
- Later architectures traded accuracy against FLOPs (Inception, MobileNet, EfficientNet) or modernised
  the ResNet (ResNeXt, ConvNeXt); the residual block is the unit everything else is built from.
- Filters, feature maps and occlusion maps make a CNN's behaviour inspectable: early layers are generic
  edge and colour detectors, deep layers are abstract, and a good model's evidence sits on the object.

## Further reading

- LeCun, Y., Bottou, L., Bengio, Y. and Haffner, P. (1998). Gradient-Based Learning Applied to Document Recognition. *Proceedings of the IEEE*. The LeNet-5 paper.
- Krizhevsky, A., Sutskever, I. and Hinton, G. E. (2012). ImageNet Classification with Deep Convolutional Neural Networks. *NeurIPS*.
- Simonyan, K. and Zisserman, A. (2015). Very Deep Convolutional Networks for Large-Scale Image Recognition. *ICLR*. (VGG)
- Szegedy, C. et al. (2015). Going Deeper with Convolutions. *CVPR*. (GoogLeNet / Inception)
- He, K., Zhang, X., Ren, S. and Sun, J. (2016). Deep Residual Learning for Image Recognition. *CVPR*.
- He, K., Zhang, X., Ren, S. and Sun, J. (2016). Identity Mappings in Deep Residual Networks. *ECCV*. (Pre-activation blocks and the gradient-flow analysis of Section 6.2)
- Ioffe, S. and Szegedy, C. (2015). Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift. *ICML*.
- Zeiler, M. D. and Fergus, R. (2014). Visualizing and Understanding Convolutional Networks. *ECCV*. (Feature-map visualisation and occlusion sensitivity)
- Dumoulin, V. and Visin, F. (2016). A Guide to Convolution Arithmetic for Deep Learning. arXiv:1603.07285. (Every output-size formula, with animations)
- Luo, W., Li, Y., Urtasun, R. and Zemel, R. (2016). Understanding the Effective Receptive Field in Deep Convolutional Neural Networks. *NeurIPS*.
- Howard, A. G. et al. (2017). MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications. arXiv:1704.04861.
- Hu, J., Shen, L. and Sun, G. (2018). Squeeze-and-Excitation Networks. *CVPR*.
- Liu, Z. et al. (2022). A ConvNet for the 2020s. *CVPR*. (ConvNeXt)
- Zhang, A., Lipton, Z. C., Li, M. and Smola, A. J. *Dive into Deep Learning*, chapters 7-8 (Convolutional Neural Networks; Modern CNNs). https://d2l.ai
- Goodfellow, I., Bengio, Y. and Courville, A. (2016). *Deep Learning*, chapter 9 (Convolutional Networks). MIT Press.
- PyTorch documentation: `torch.nn.Conv2d`, `torch.nn.functional.conv2d`, `torchvision.transforms.v2`. https://docs.pytorch.org/docs/stable/ and https://docs.pytorch.org/vision/stable/